In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/project4/sample_submission.csv
/kaggle/input/competitions/project4/train.csv
/kaggle/input/competitions/project4/test.csv
/kaggle/input/competitions/project4/audio/audio/test/597524f0.wav
/kaggle/input/competitions/project4/audio/audio/test/ca2959f3.wav
/kaggle/input/competitions/project4/audio/audio/test/df9b2d9b.wav
/kaggle/input/competitions/project4/audio/audio/test/ba878534.wav
/kaggle/input/competitions/project4/audio/audio/test/788adecc.wav
/kaggle/input/competitions/project4/audio/audio/test/42579f19.wav
/kaggle/input/competitions/project4/audio/audio/test/c1be56b2.wav
/kaggle/input/competitions/project4/audio/audio/test/fdc5bc04.wav
/kaggle/input/competitions/project4/audio/audio/test/4d3ddfd8.wav
/kaggle/input/competitions/project4/audio/audio/test/d8b06123.wav
/kaggle/input/competitions/project4/audio/audio/test/ea763cbc.wav
/kaggle/input/competitions/project4/audio/audio/test/6e1bfe16.wav
/kaggle/input/competitions/project4/audio/audio/test/af0ce1


# P4 — Listening Past the Noise
### Audio Classification Using Classical Machine Learning

## 1. Project Overview

The objective of this project is to classify short audio recordings into one of 41 predefined categories.

The task requires predicting the three most likely categories for each test recording, ranked in descending order of confidence.

## 2. Evaluation Metric

The competition uses Mean Average Precision at 3 (MAP@3).

- Correct prediction at rank 1: 1 point
- Correct prediction at rank 2: 1/2 point
- Correct prediction at rank 3: 1/3 point
- Correct category outside the top three: 0 points

## 3. Methodology

Our approach consists of:

1. Dataset inspection and label-quality analysis.
2. Audio feature extraction using Librosa.
3. Training and comparison of classical machine learning models.
4. Hyperparameter tuning and cross-validation.
5. Investigation of noisy and unchecked labels.
6. Feature engineering and selection.
7. Probability-based ensemble learning.
8. Final test prediction generation and submission validation.


Task 1 — verify the dataset

In [2]:

from pathlib import Path
import pandas as pd
import os

# Kaggle input directory
INPUT_ROOT = Path("/kaggle/input")

print("DATASETS AVAILABLE:")
for item in INPUT_ROOT.iterdir():
    print(item.name)

# Locate the competition CSV files
train_files = list(INPUT_ROOT.rglob("train.csv"))
test_files = list(INPUT_ROOT.rglob("test.csv"))
sample_files = list(INPUT_ROOT.rglob("sample_submission.csv"))

print("\nCSV FILE LOCATIONS:")
print("Train:", train_files)
print("Test:", test_files)
print("Sample submission:", sample_files)

# Make sure each required CSV was found exactly once
assert len(train_files) == 1, "Expected exactly one train.csv"
assert len(test_files) == 1, "Expected exactly one test.csv"
assert len(sample_files) == 1, "Expected exactly one sample_submission.csv"

# Load the CSV files
train_df = pd.read_csv(train_files[0])
test_df = pd.read_csv(test_files[0])
sample_df = pd.read_csv(sample_files[0])

print("\nDATASET SHAPES:")
print("Training data:", train_df.shape)
print("Test data:", test_df.shape)
print("Sample submission:", sample_df.shape)

print("\nTRAINING DATA:")
display(train_df.head())

print("\nTEST DATA:")
display(test_df.head())

print("\nSAMPLE SUBMISSION:")
display(sample_df.head())

# Locate audio directories
audio_train_dirs = list(INPUT_ROOT.rglob("audio/train"))
audio_test_dirs = list(INPUT_ROOT.rglob("audio/test"))

print("\nAUDIO FOLDERS:")
print("Training audio folders:", audio_train_dirs)
print("Test audio folders:", audio_test_dirs)

assert len(audio_train_dirs) == 1, "Could not uniquely locate audio/train"
assert len(audio_test_dirs) == 1, "Could not uniquely locate audio/test"

# Count WAV recordings
train_audio_count = sum(
    1 for _ in audio_train_dirs[0].glob("*.wav")
)

test_audio_count = sum(
    1 for _ in audio_test_dirs[0].glob("*.wav")
)

print("\nAUDIO FILE COUNTS:")
print("Training audio files:", train_audio_count)
print("Test audio files:", test_audio_count)

print("\nFIRST 5 TRAINING AUDIO FILES:")
print([p.name for p in list(audio_train_dirs[0].glob("*.wav"))[:5]])


DATASETS AVAILABLE:
competitions

CSV FILE LOCATIONS:
Train: [PosixPath('/kaggle/input/competitions/project4/train.csv')]
Test: [PosixPath('/kaggle/input/competitions/project4/test.csv')]
Sample submission: [PosixPath('/kaggle/input/competitions/project4/sample_submission.csv')]

DATASET SHAPES:
Training data: (5604, 3)
Test data: (1592, 1)
Sample submission: (1592, 2)

TRAINING DATA:


,id,label,checked
0,581a44c1,C16,0
1,6e3fbae8,C04,0
2,4d8a0977,C14,0
3,5a44abb6,C06,0
4,d337c44b,C30,0



TEST DATA:


,id
0,c99ed9e0
1,f7988da2
2,af43e3af
3,fbb23ad8
4,3be2034b



SAMPLE SUBMISSION:


,id,label
0,c99ed9e0,C01 C02 C03
1,f7988da2,C01 C02 C03
2,af43e3af,C01 C02 C03
3,fbb23ad8,C01 C02 C03
4,3be2034b,C01 C02 C03



AUDIO FOLDERS:
Training audio folders: [PosixPath('/kaggle/input/competitions/project4/audio/audio/train')]
Test audio folders: [PosixPath('/kaggle/input/competitions/project4/audio/audio/test')]

AUDIO FILE COUNTS:
Training audio files: 5604
Test audio files: 1592

FIRST 5 TRAINING AUDIO FILES:
['4e2f0a5d.wav', 'b2b408f4.wav', 'f3c74d1b.wav', '3348a7cd.wav', '37b97628.wav']


The labels and data quality

In [3]:

# STEP 2: Inspect label distribution and data quality

import pandas as pd

# 1. Basic data-quality checks
print("MISSING VALUES:")
print(train_df.isnull().sum())

print("\nDUPLICATE TRAINING IDs:")
print(train_df["id"].duplicated().sum())

print("\nDUPLICATE TEST IDs:")
print(test_df["id"].duplicated().sum())

# 2. Number of unique categories
categories = sorted(train_df["label"].unique())

print("\nNUMBER OF CATEGORIES:", len(categories))
print("CATEGORY LABELS:")
print(categories)

# 3. Overall category distribution
print("\nTOTAL SAMPLES PER CATEGORY:")
category_counts = train_df["label"].value_counts().sort_index()
display(category_counts.to_frame(name="total_samples"))

# 4. Verified and unverified samples
print("\nCHECKED COLUMN DISTRIBUTION:")
print(train_df["checked"].value_counts().sort_index())

# 5. Category-wise breakdown of checked and unchecked samples
print("\nCATEGORY-WISE LABEL RELIABILITY:")
label_quality = pd.crosstab(
    train_df["label"],
    train_df["checked"]
).reindex(columns=[0, 1], fill_value=0)

label_quality.columns = ["unchecked_0", "verified_1"]
label_quality["total"] = (
    label_quality["unchecked_0"] + label_quality["verified_1"]
)

display(label_quality)

# 6. Verify whether checked samples are equally distributed
print("\nVERIFIED SAMPLES PER CATEGORY:")
print(label_quality["verified_1"].value_counts().sort_index())

# 7. Check for unexpected checked values
print("\nUNIQUE VALUES IN CHECKED COLUMN:")
print(sorted(train_df["checked"].dropna().unique()))

# 8. Summary
print("\nSUMMARY:")
print("Total training samples:", len(train_df))
print("Total categories:", len(categories))
print("Verified samples:", (train_df["checked"] == 1).sum())
print("Unverified samples:", (train_df["checked"] == 0).sum())


MISSING VALUES:
id         0
label      0
checked    0
dtype: int64

DUPLICATE TRAINING IDs:
0

DUPLICATE TEST IDs:
0

NUMBER OF CATEGORIES: 41
CATEGORY LABELS:
['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'C25', 'C26', 'C27', 'C28', 'C29', 'C30', 'C31', 'C32', 'C33', 'C34', 'C35', 'C36', 'C37', 'C38', 'C39', 'C40', 'C41']

TOTAL SAMPLES PER CATEGORY:


,total_samples
label,
C01,172
C02,58
C03,42
C04,224
C05,80
C06,89
C07,216
C08,62
C09,181



CHECKED COLUMN DISTRIBUTION:
checked
0    4984
1     620
Name: count, dtype: int64

CATEGORY-WISE LABEL RELIABILITY:


,unchecked_0,verified_1,total
label,,,
C01,162,10,172
C02,48,10,58
C03,2,40,42
C04,214,10,224
C05,40,40,80
C06,79,10,89
C07,206,10,216
C08,52,10,62
C09,171,10,181



VERIFIED SAMPLES PER CATEGORY:
verified_1
10    34
40     7
Name: count, dtype: int64

UNIQUE VALUES IN CHECKED COLUMN:
[np.int64(0), np.int64(1)]

SUMMARY:
Total training samples: 5604
Total categories: 41
Verified samples: 620
Unverified samples: 4984


Step 3 — Inspect the audio recordings

In [4]:

# STEP 3: Inspect audio metadata

import soundfile as sf
import numpy as np
import pandas as pd

# Use the audio directories discovered in Step 1
train_audio_dir = audio_train_dirs[0]
test_audio_dir = audio_test_dirs[0]

def inspect_audio_folder(folder, name):
    records = []
    errors = []

    wav_files = list(folder.glob("*.wav"))

    print(f"\nInspecting {name} audio...")
    print("Number of WAV files:", len(wav_files))

    for file_path in wav_files:
        try:
            info = sf.info(str(file_path))

            records.append({
                "id": file_path.stem,
                "sample_rate": info.samplerate,
                "channels": info.channels,
                "duration": info.duration,
                "frames": info.frames,
                "subtype": info.subtype,
                "format": info.format
            })

        except Exception as error:
            errors.append((file_path.name, str(error)))

    metadata = pd.DataFrame(records)

    print("\nSuccessfully inspected:", len(metadata))
    print("Unreadable files:", len(errors))

    if len(metadata) > 0:
        print("\nSAMPLE RATES:")
        print(metadata["sample_rate"].value_counts())

        print("\nNUMBER OF CHANNELS:")
        print(metadata["channels"].value_counts())

        print("\nAUDIO SUBTYPES:")
        print(metadata["subtype"].value_counts())

        print("\nDURATION STATISTICS (seconds):")
        print(metadata["duration"].describe())

        print("\nMINIMUM DURATION:")
        print(metadata["duration"].min())

        print("\nMAXIMUM DURATION:")
        print(metadata["duration"].max())

    if errors:
        print("\nFIRST 5 UNREADABLE FILES:")
        print(errors[:5])

    return metadata, errors


# Inspect training and test audio
train_metadata, train_errors = inspect_audio_folder(
    train_audio_dir, "Training"
)

test_metadata, test_errors = inspect_audio_folder(
    test_audio_dir, "Test"
)

# Check whether CSV IDs match audio filenames
train_csv_ids = set(train_df["id"].astype(str))
test_csv_ids = set(test_df["id"].astype(str))

train_audio_ids = set(train_metadata["id"])
test_audio_ids = set(test_metadata["id"])

print("\nID AND AUDIO FILE MATCHING:")
print("Training IDs without audio:",
      len(train_csv_ids - train_audio_ids))

print("Training audio without CSV ID:",
      len(train_audio_ids - train_csv_ids))

print("Test IDs without audio:",
      len(test_csv_ids - test_audio_ids))

print("Test audio without CSV ID:",
      len(test_audio_ids - test_csv_ids))



Inspecting Training audio...
Number of WAV files: 5604

Successfully inspected: 5604
Unreadable files: 0

SAMPLE RATES:
sample_rate
22050    5604
Name: count, dtype: int64

NUMBER OF CHANNELS:
channels
1    5604
Name: count, dtype: int64

AUDIO SUBTYPES:
subtype
PCM_16    5604
Name: count, dtype: int64

DURATION STATISTICS (seconds):
count    5604.000000
mean        7.880497
std         7.566021
min         0.375329
25%         2.023095
50%         4.932880
75%        11.360113
max        31.437823
Name: duration, dtype: float64

MINIMUM DURATION:
0.37532879818594106

MAXIMUM DURATION:
31.4378231292517

Inspecting Test audio...
Number of WAV files: 1592

Successfully inspected: 1592
Unreadable files: 0

SAMPLE RATES:
sample_rate
22050    1592
Name: count, dtype: int64

NUMBER OF CHANNELS:
channels
1    1592
Name: count, dtype: int64

AUDIO SUBTYPES:
subtype
PCM_16    1592
Name: count, dtype: int64

DURATION STATISTICS (seconds):
count    1592.000000
mean        5.491405
std         5.

STEP 4.1 Install/import and define the extractor

In [5]:

# STEP 4: Audio feature extraction

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path
import time

print("Librosa version:", librosa.__version__)

# Feature extraction settings
SAMPLE_RATE = 22050
N_MFCC = 20
N_FFT = 2048
HOP_LENGTH = 512


def summarize_feature(feature):
    """
    Calculate mean and standard deviation
    across the time frames of a feature.
    """
    return np.concatenate([
        np.mean(feature, axis=1),
        np.std(feature, axis=1)
    ])


def extract_features(file_path):
    """
    Extract a fixed-length feature vector
    from one audio recording.
    """

    audio, sr = sf.read(str(file_path))

    # All recordings should already be mono.
    # This is a safety check.
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    audio = audio.astype(np.float32)

    features = []

    # 1. MFCC features
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )

    features.extend(summarize_feature(mfcc))

    # 2. Chroma features
    chroma = librosa.feature.chroma_stft(
        y=audio,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )

    features.extend(summarize_feature(chroma))

    # 3. Spectral features
    spectral_features = [
        librosa.feature.spectral_centroid(
            y=audio, sr=sr, n_fft=N_FFT,
            hop_length=HOP_LENGTH
        ),
        librosa.feature.spectral_bandwidth(
            y=audio, sr=sr, n_fft=N_FFT,
            hop_length=HOP_LENGTH
        ),
        librosa.feature.spectral_rolloff(
            y=audio, sr=sr, n_fft=N_FFT,
            hop_length=HOP_LENGTH
        ),
        librosa.feature.zero_crossing_rate(
            y=audio, frame_length=N_FFT,
            hop_length=HOP_LENGTH
        ),
        librosa.feature.rms(
            y=audio, frame_length=N_FFT,
            hop_length=HOP_LENGTH
        )
    ]

    for feature in spectral_features:
        features.extend(summarize_feature(feature))

    # 4. Duration
    duration = len(audio) / sr
    features.append(duration)

    return np.asarray(features, dtype=np.float32)


def extract_folder_features(folder, name):
    """
    Extract features from every WAV file in a folder.
    """

    wav_files = sorted(folder.glob("*.wav"))

    feature_rows = []
    ids = []
    errors = []

    start_time = time.time()

    for i, file_path in enumerate(wav_files):

        try:
            feature_vector = extract_features(file_path)

            feature_rows.append(feature_vector)
            ids.append(file_path.stem)

        except Exception as error:
            errors.append((file_path.name, str(error)))

        if (i + 1) % 500 == 0:
            print(
                f"{name}: processed {i + 1}/{len(wav_files)} files"
            )

    X = np.vstack(feature_rows)

    elapsed = time.time() - start_time

    print(f"\n{name} extraction completed.")
    print("Feature matrix shape:", X.shape)
    print("Successful files:", len(ids))
    print("Failed files:", len(errors))
    print("Time taken (seconds):", round(elapsed, 2))

    if errors:
        print("First 5 errors:", errors[:5])

    return X, ids, errors


print("Feature extractor defined successfully.")


Librosa version: 0.11.0
Feature extractor defined successfully.


STEP 4.2 Run feature extraction

In [6]:

# Extract features from training recordings
X_train, train_feature_ids, train_feature_errors = (
    extract_folder_features(train_audio_dir, "Training")
)

# Extract features from test recordings
X_test, test_feature_ids, test_feature_errors = (
    extract_folder_features(test_audio_dir, "Test")
)

print("\nFINAL FEATURE MATRIX SHAPES")
print("Training:", X_train.shape)
print("Test:", X_test.shape)

# Save extracted features so we do not need to
# process all WAV files again if the session restarts.
np.savez_compressed(
    "/kaggle/working/audio_features.npz",
    X_train=X_train,
    X_test=X_test,
    train_ids=np.array(train_feature_ids),
    test_ids=np.array(test_feature_ids)
)

print("\nFeatures saved to /kaggle/working/audio_features.npz")


Training: processed 500/5604 files
Training: processed 1000/5604 files
Training: processed 1500/5604 files


/usr/local/lib/python3.13/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


Training: processed 2000/5604 files
Training: processed 2500/5604 files
Training: processed 3000/5604 files
Training: processed 3500/5604 files
Training: processed 4000/5604 files
Training: processed 4500/5604 files
Training: processed 5000/5604 files
Training: processed 5500/5604 files

Training extraction completed.
Feature matrix shape: (5604, 75)
Successful files: 5604
Failed files: 0
Time taken (seconds): 690.36
Test: processed 500/1592 files
Test: processed 1000/1592 files
Test: processed 1500/1592 files

Test extraction completed.
Feature matrix shape: (1592, 75)
Successful files: 1592
Failed files: 0
Time taken (seconds): 135.72

FINAL FEATURE MATRIX SHAPES
Training: (5604, 75)
Test: (1592, 75)

Features saved to /kaggle/working/audio_features.npz


Step 5 — Verify feature-to-label alignment

In [7]:

# STEP 5: Align extracted features with CSV rows

# 1. Create feature tables using recording IDs
train_features_df = pd.DataFrame(
    X_train,
    index=train_feature_ids
)

test_features_df = pd.DataFrame(
    X_test,
    index=test_feature_ids
)

# 2. Reorder features to match the original CSV order
X_train_aligned = train_features_df.reindex(
    train_df["id"].astype(str)
).to_numpy()

X_test_aligned = test_features_df.reindex(
    test_df["id"].astype(str)
).to_numpy()

# 3. Check alignment
print("TRAINING ALIGNMENT:")
print("Expected rows:", len(train_df))
print("Actual rows:", X_train_aligned.shape[0])
print("Missing feature rows:", np.isnan(X_train_aligned).any(axis=1).sum())

print("\nTEST ALIGNMENT:")
print("Expected rows:", len(test_df))
print("Actual rows:", X_test_aligned.shape[0])
print("Missing feature rows:", np.isnan(X_test_aligned).any(axis=1).sum())

# 4. Check for invalid numerical values
print("\nNUMERICAL QUALITY:")
print("Training NaN values:", np.isnan(X_train_aligned).sum())
print("Training infinite values:", np.isinf(X_train_aligned).sum())

print("Test NaN values:", np.isnan(X_test_aligned).sum())
print("Test infinite values:", np.isinf(X_test_aligned).sum())

# 5. Check feature variation
feature_std = np.std(X_train_aligned, axis=0)

print("\nFEATURE VARIATION:")
print("Constant features:", np.sum(feature_std == 0))
print("Minimum feature standard deviation:", feature_std.min())
print("Maximum feature standard deviation:", feature_std.max())

# 6. Create aligned training and test data
# Labels now correspond to the correct feature rows.
y_train = train_df["label"].to_numpy()
checked_train = train_df["checked"].to_numpy()

assert X_train_aligned.shape[0] == len(y_train)
assert X_test_aligned.shape[0] == len(test_df)

assert np.isfinite(X_train_aligned).all()
assert np.isfinite(X_test_aligned).all()

# 7. Save aligned features
np.savez_compressed(
    "/kaggle/working/aligned_audio_features.npz",
    X_train=X_train_aligned,
    X_test=X_test_aligned,
    y_train=y_train,
    checked=checked_train,
    train_ids=train_df["id"].astype(str).to_numpy(),
    test_ids=test_df["id"].astype(str).to_numpy()
)

print("\nSUCCESS: Features are aligned with CSV rows.")
print("Aligned feature file saved.")


TRAINING ALIGNMENT:
Expected rows: 5604
Actual rows: 5604
Missing feature rows: 0

TEST ALIGNMENT:
Expected rows: 1592
Actual rows: 1592
Missing feature rows: 0

NUMERICAL QUALITY:
Training NaN values: 0
Training infinite values: 0
Test NaN values: 0
Test infinite values: 0

FEATURE VARIATION:
Constant features: 0
Minimum feature standard deviation: 0.04161888
Maximum feature standard deviation: 1981.8536

SUCCESS: Features are aligned with CSV rows.
Aligned feature file saved.


Step 6 — Establish a reliable validation strategy

In [8]:

# STEP 6: Verified-only baseline with 5-fold cross-validation

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. Select only verified training recordings
verified_mask = (checked_train == 1)

X_verified = X_train_aligned[verified_mask]
y_verified = y_train[verified_mask]

print("VERIFIED DATASET")
print("Feature matrix:", X_verified.shape)
print("Labels:", y_verified.shape)
print("Number of categories:", len(np.unique(y_verified)))

# 2. Define the model pipeline
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=3000,
        random_state=42
    ))
])

# 3. Define 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 4. Custom MAP@3 calculation
def calculate_map3(y_true, probabilities, class_labels):

    # Sort category probabilities from highest to lowest
    top3_indices = np.argsort(
        probabilities, axis=1
    )[:, ::-1][:, :3]

    top3_labels = class_labels[top3_indices]

    scores = []

    for actual, predictions in zip(y_true, top3_labels):

        if actual == predictions[0]:
            scores.append(1.0)

        elif actual == predictions[1]:
            scores.append(0.5)

        elif actual == predictions[2]:
            scores.append(1.0 / 3.0)

        else:
            scores.append(0.0)

    return np.mean(scores)


# 5. Generate out-of-fold predictions
classes = np.sort(np.unique(y_verified))

oof_probabilities = np.zeros(
    (len(y_verified), len(classes))
)

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_verified, y_verified), start=1
):

    X_fold_train = X_verified[train_idx]
    y_fold_train = y_verified[train_idx]

    X_fold_val = X_verified[val_idx]
    y_fold_val = y_verified[val_idx]

    # Train using only the fold's training portion
    baseline_model.fit(
        X_fold_train,
        y_fold_train
    )

    # Predict probabilities for the held-out fold
    fold_probabilities = baseline_model.predict_proba(
        X_fold_val
    )

    # Place probabilities in the common class order
    fold_classes = baseline_model.named_steps[
        "classifier"
    ].classes_

    for j, label in enumerate(fold_classes):
        class_position = np.where(classes == label)[0][0]

        oof_probabilities[val_idx, class_position] = (
            fold_probabilities[:, j]
        )

    # Calculate MAP@3 for this fold
    fold_map3 = calculate_map3(
        y_fold_val,
        oof_probabilities[val_idx],
        classes
    )

    fold_scores.append(fold_map3)

    print(
        f"Fold {fold}: "
        f"MAP@3 = {fold_map3:.4f}"
    )


# 6. Overall out-of-fold MAP@3
overall_map3 = calculate_map3(
    y_verified,
    oof_probabilities,
    classes
)

print("\nBASELINE RESULTS")
print("Fold scores:", np.round(fold_scores, 4))
print("Mean fold MAP@3:", round(np.mean(fold_scores), 4))
print("Overall OOF MAP@3:", round(overall_map3, 4))


VERIFIED DATASET
Feature matrix: (620, 75)
Labels: (620,)
Number of categories: 41
Fold 1: MAP@3 = 0.6129
Fold 2: MAP@3 = 0.6626
Fold 3: MAP@3 = 0.6653
Fold 4: MAP@3 = 0.6653
Fold 5: MAP@3 = 0.6290

BASELINE RESULTS
Fold scores: [0.6129 0.6626 0.6653 0.6653 0.629 ]
Mean fold MAP@3: 0.647
Overall OOF MAP@3: 0.647


Step 7 — Compare four classical ML models

In [9]:

# STEP 7: Compare classical ML models

from sklearn.base import clone
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
import time

# Define four candidate models
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        ))
    ]),

    "RBF SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            C=3.0,
            kernel="rbf",
            class_weight="balanced",
            probability=True,
            random_state=42
        ))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        class_weight="balanced",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=400,
        class_weight="balanced",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

# Store results
comparison_results = []

for model_name, model in models.items():

    print("\n" + "=" * 55)
    print("MODEL:", model_name)
    print("=" * 55)

    oof = np.zeros((len(y_verified), len(classes)))
    fold_scores = []

    start = time.time()

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X_verified, y_verified), start=1
    ):

        X_fold_train = X_verified[train_idx]
        y_fold_train = y_verified[train_idx]

        X_fold_val = X_verified[val_idx]
        y_fold_val = y_verified[val_idx]

        current_model = clone(model)

        current_model.fit(
            X_fold_train,
            y_fold_train
        )

        probabilities = current_model.predict_proba(
            X_fold_val
        )

        model_classes = current_model.classes_ if hasattr(
            current_model, "classes_"
        ) else current_model.named_steps[
            "classifier"
        ].classes_

        # Align probability columns with common class order
        for j, label in enumerate(model_classes):
            position = np.where(classes == label)[0][0]
            oof[val_idx, position] = probabilities[:, j]

        score = calculate_map3(
            y_fold_val,
            oof[val_idx],
            classes
        )

        fold_scores.append(score)

        print(f"Fold {fold}: MAP@3 = {score:.4f}")

    overall_score = calculate_map3(
        y_verified,
        oof,
        classes
    )

    elapsed = time.time() - start

    comparison_results.append({
        "Model": model_name,
        "Mean Fold MAP@3": np.mean(fold_scores),
        "Overall OOF MAP@3": overall_score,
        "Time (seconds)": elapsed
    })

    print("Overall OOF MAP@3:", round(overall_score, 4))
    print("Time taken:", round(elapsed, 2), "seconds")


# Display comparison
comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.sort_values(
    "Overall OOF MAP@3",
    ascending=False
).reset_index(drop=True)

print("\nFINAL MODEL COMPARISON:")
display(comparison_df)



MODEL: Logistic Regression
Fold 1: MAP@3 = 0.6129
Fold 2: MAP@3 = 0.6626
Fold 3: MAP@3 = 0.6653
Fold 4: MAP@3 = 0.6653
Fold 5: MAP@3 = 0.6290
Overall OOF MAP@3: 0.647
Time taken: 7.5 seconds

MODEL: RBF SVM
Fold 1: MAP@3 = 0.6505
Fold 2: MAP@3 = 0.6653
Fold 3: MAP@3 = 0.7124
Fold 4: MAP@3 = 0.6519
Fold 5: MAP@3 = 0.6505
Overall OOF MAP@3: 0.6661
Time taken: 1.61 seconds

MODEL: Random Forest
Fold 1: MAP@3 = 0.6653
Fold 2: MAP@3 = 0.6331
Fold 3: MAP@3 = 0.6774
Fold 4: MAP@3 = 0.6586
Fold 5: MAP@3 = 0.6022
Overall OOF MAP@3: 0.6473
Time taken: 7.08 seconds

MODEL: Extra Trees
Fold 1: MAP@3 = 0.6452
Fold 2: MAP@3 = 0.6277
Fold 3: MAP@3 = 0.6882
Fold 4: MAP@3 = 0.6492
Fold 5: MAP@3 = 0.6102
Overall OOF MAP@3: 0.6441
Time taken: 4.54 seconds

FINAL MODEL COMPARISON:


,Model,Mean Fold MAP@3,Overall OOF MAP@3,Time (seconds)
0,RBF SVM,0.666129,0.666129,1.607394
1,Random Forest,0.647312,0.647312,7.078400
2,Logistic Regression,0.647043,0.647043,7.501511
3,Extra Trees,0.644086,0.644086,4.542679


Step 8 — Tune the RBF SVM

In [10]:

# STEP 8: RBF SVM hyperparameter comparison

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import time

svm_configs = []

C_values = [0.5, 1.0, 3.0, 10.0]
class_weight_values = [None, "balanced"]

for C_value in C_values:
    for weight_value in class_weight_values:

        svm_configs.append({
            "C": C_value,
            "class_weight": weight_value
        })


svm_results = []

for config in svm_configs:

    print("\nTesting:", config)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            C=config["C"],
            kernel="rbf",
            gamma="scale",
            class_weight=config["class_weight"],
            probability=True,
            random_state=42
        ))
    ])

    oof = np.zeros((len(y_verified), len(classes)))
    fold_scores = []

    start = time.time()

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X_verified, y_verified), start=1
    ):

        current_model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", SVC(
                C=config["C"],
                kernel="rbf",
                gamma="scale",
                class_weight=config["class_weight"],
                probability=True,
                random_state=42
            ))
        ])

        current_model.fit(
            X_verified[train_idx],
            y_verified[train_idx]
        )

        probabilities = current_model.predict_proba(
            X_verified[val_idx]
        )

        model_classes = current_model.named_steps[
            "classifier"
        ].classes_

        for j, label in enumerate(model_classes):
            position = np.where(classes == label)[0][0]
            oof[val_idx, position] = probabilities[:, j]

        fold_score = calculate_map3(
            y_verified[val_idx],
            oof[val_idx],
            classes
        )

        fold_scores.append(fold_score)

    overall_score = calculate_map3(
        y_verified,
        oof,
        classes
    )

    elapsed = time.time() - start

    svm_results.append({
        "C": config["C"],
        "class_weight": str(config["class_weight"]),
        "Mean Fold MAP@3": np.mean(fold_scores),
        "Overall OOF MAP@3": overall_score,
        "Fold Scores": np.round(fold_scores, 4),
        "Time (seconds)": round(elapsed, 2)
    })

    print("Fold scores:", np.round(fold_scores, 4))
    print("Overall OOF MAP@3:", round(overall_score, 4))


svm_results_df = pd.DataFrame(svm_results)

svm_results_df = svm_results_df.sort_values(
    "Overall OOF MAP@3",
    ascending=False
).reset_index(drop=True)

print("\nSVM HYPERPARAMETER RESULTS:")
display(svm_results_df)



Testing: {'C': 0.5, 'class_weight': None}
Fold scores: [0.5363 0.5323 0.5188 0.5376 0.5255]
Overall OOF MAP@3: 0.5301

Testing: {'C': 0.5, 'class_weight': 'balanced'}
Fold scores: [0.5215 0.5255 0.4933 0.5094 0.4973]
Overall OOF MAP@3: 0.5094

Testing: {'C': 1.0, 'class_weight': None}
Fold scores: [0.6142 0.6505 0.6263 0.6304 0.6263]
Overall OOF MAP@3: 0.6296

Testing: {'C': 1.0, 'class_weight': 'balanced'}
Fold scores: [0.6358 0.625  0.6613 0.6237 0.625 ]
Overall OOF MAP@3: 0.6341

Testing: {'C': 3.0, 'class_weight': None}
Fold scores: [0.6465 0.6505 0.7083 0.6653 0.6573]
Overall OOF MAP@3: 0.6656

Testing: {'C': 3.0, 'class_weight': 'balanced'}
Fold scores: [0.6505 0.6653 0.7124 0.6519 0.6505]
Overall OOF MAP@3: 0.6661

Testing: {'C': 10.0, 'class_weight': None}
Fold scores: [0.6519 0.6667 0.7137 0.6667 0.6546]
Overall OOF MAP@3: 0.6707

Testing: {'C': 10.0, 'class_weight': 'balanced'}
Fold scores: [0.6492 0.6667 0.711  0.672  0.6505]
Overall OOF MAP@3: 0.6699

SVM HYPERPARAMETER RE

,C,class_weight,Mean Fold MAP@3,Overall OOF MAP@3,Fold Scores,Time (seconds)
0,10.0,None,0.670699,0.670699,"[0.6519, 0.6667, 0.7137, 0.6667, 0.6546]",1.52
1,10.0,balanced,0.669892,0.669892,"[0.6492, 0.6667, 0.711, 0.672, 0.6505]",1.54
2,3.0,balanced,0.666129,0.666129,"[0.6505, 0.6653, 0.7124, 0.6519, 0.6505]",1.51
3,3.0,None,0.665591,0.665591,"[0.6465, 0.6505, 0.7083, 0.6653, 0.6573]",1.51
4,1.0,balanced,0.634140,0.634140,"[0.6358, 0.625, 0.6613, 0.6237, 0.625]",1.53
5,1.0,None,0.629570,0.629570,"[0.6142, 0.6505, 0.6263, 0.6304, 0.6263]",1.45
6,0.5,None,0.530108,0.530108,"[0.5363, 0.5323, 0.5188, 0.5376, 0.5255]",1.41
7,0.5,balanced,0.509409,0.509409,"[0.5215, 0.5255, 0.4933, 0.5094, 0.4973]",1.56


Test higher C values

In [11]:

C_values = [10.0, 20.0, 30.0, 50.0]

high_c_results = []

for C_value in C_values:

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            C=C_value,
            kernel="rbf",
            gamma="scale",
            class_weight=None,
            probability=True,
            random_state=42
        ))
    ])

    fold_scores = []
    oof_probabilities = np.zeros(
        (len(y_verified), len(np.unique(y_verified)))
    )

    start_time = time.time()

    for train_idx, val_idx in cv.split(X_verified, y_verified):

        X_fold_train = X_verified[train_idx]
        y_fold_train = y_verified[train_idx]

        X_fold_val = X_verified[val_idx]
        y_fold_val = y_verified[val_idx]

        model.fit(X_fold_train, y_fold_train)

        probabilities = model.predict_proba(X_fold_val)

        oof_probabilities[val_idx] = probabilities

        score = calculate_map3(
            y_fold_val,
            probabilities,
            model.named_steps["svm"].classes_
        )

        fold_scores.append(score)

    overall_score = calculate_map3(
        y_verified,
        oof_probabilities,
        model.named_steps["svm"].classes_
    )

    high_c_results.append({
        "C": C_value,
        "Mean Fold MAP@3": np.mean(fold_scores),
        "Overall OOF MAP@3": overall_score,
        "Fold Scores": np.round(fold_scores, 4).tolist(),
        "Time (seconds)": round(time.time() - start_time, 2)
    })

high_c_df = pd.DataFrame(high_c_results)

display(
    high_c_df.sort_values(
        "Overall OOF MAP@3",
        ascending=False
    ).reset_index(drop=True)
)


,C,Mean Fold MAP@3,Overall OOF MAP@3,Fold Scores,Time (seconds)
0,10.0,0.670699,0.670699,"[0.6519, 0.6667, 0.7137, 0.6667, 0.6546]",1.53
1,20.0,0.670161,0.670161,"[0.6478, 0.668, 0.7137, 0.6667, 0.6546]",1.53
2,30.0,0.670161,0.670161,"[0.6478, 0.668, 0.7137, 0.6667, 0.6546]",1.51
3,50.0,0.670161,0.670161,"[0.6478, 0.668, 0.7137, 0.6667, 0.6546]",1.53


Step 9 — Predict and inspect unchecked labels

In [12]:

# Separate the unchecked training samples
unchecked_mask = checked_train == 0

X_unchecked = X_train_aligned[unchecked_mask]

unchecked_df = train_df.loc[
    unchecked_mask, ["id", "label", "checked"]
].copy()

# Build our current best SVM model
best_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        C=10.0,
        kernel="rbf",
        gamma="scale",
        class_weight=None,
        probability=True,
        random_state=42
    ))
])

# Train only on verified samples
best_svm.fit(X_verified, y_verified)

# Predict the unchecked samples
unchecked_probabilities = best_svm.predict_proba(X_unchecked)

class_labels = best_svm.named_steps["svm"].classes_

predicted_indices = np.argmax(
    unchecked_probabilities, axis=1
)

predicted_labels = class_labels[predicted_indices]

# Calculate the model's maximum predicted probability
confidence_scores = np.max(
    unchecked_probabilities, axis=1
)

# Store predictions alongside the existing labels
unchecked_df["predicted_label"] = predicted_labels
unchecked_df["confidence"] = confidence_scores

unchecked_df["agrees_with_existing_label"] = (
    unchecked_df["label"] == unchecked_df["predicted_label"]
)

# Overall agreement
agreement = unchecked_df[
    "agrees_with_existing_label"
].mean()

print("Unchecked samples:", len(unchecked_df))
print("Model agreement with existing labels:", round(agreement, 4))
print("Agreement percentage:", round(agreement * 100, 2), "%")

# Agreement at different confidence levels
unchecked_df["confidence_group"] = pd.cut(
    unchecked_df["confidence"],
    bins=[0, 0.4, 0.6, 0.8, 1.0],
    include_lowest=True
)

confidence_summary = unchecked_df.groupby(
    "confidence_group",
    observed=False
).agg(
    sample_count=("id", "count"),
    agreement_rate=("agrees_with_existing_label", "mean"),
    average_confidence=("confidence", "mean")
).reset_index()

display(confidence_summary)

# Inspect a few examples
display(unchecked_df.head(10))


Unchecked samples: 4984
Model agreement with existing labels: 0.2596
Agreement percentage: 25.96 %


,confidence_group,sample_count,agreement_rate,average_confidence
0,"(-0.001, 0.4]",4604,0.234796,0.196672
1,"(0.4, 0.6]",346,0.540462,0.464525
2,"(0.6, 0.8]",29,0.724138,0.659212
3,"(0.8, 1.0]",5,1.000000,0.855095


,id,label,checked,predicted_label,confidence,agrees_with_existing_label,confidence_group
0,581a44c1,C16,0,C16,0.240961,True,"(-0.001, 0.4]"
1,6e3fbae8,C04,0,C12,0.134776,False,"(-0.001, 0.4]"
2,4d8a0977,C14,0,C25,0.179172,False,"(-0.001, 0.4]"
3,5a44abb6,C06,0,C37,0.336710,False,"(-0.001, 0.4]"
4,d337c44b,C30,0,C05,0.275181,False,"(-0.001, 0.4]"
5,30bdae9b,C25,0,C25,0.332794,True,"(-0.001, 0.4]"
6,823c6825,C25,0,C38,0.401459,False,"(0.4, 0.6]"
7,d32243d6,C28,0,C38,0.133699,False,"(-0.001, 0.4]"
8,f0c97231,C21,0,C05,0.159620,False,"(-0.001, 0.4]"
9,d58f70a0,C17,0,C17,0.206722,True,"(-0.001, 0.4]"


Step 10 — Validate whether SVM confidence is trustworthy

In [13]:

# Store out-of-fold predictions and confidence for verified samples
oof_predictions = np.empty(len(y_verified), dtype=object)
oof_confidence = np.zeros(len(y_verified))

for train_idx, val_idx in cv.split(X_verified, y_verified):

    # Create a fresh SVM for this fold
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            C=10.0,
            kernel="rbf",
            gamma="scale",
            class_weight=None,
            probability=True,
            random_state=42
        ))
    ])

    # Train on the fold's training portion
    model.fit(
        X_verified[train_idx],
        y_verified[train_idx]
    )

    # Predict probabilities for unseen validation samples
    probabilities = model.predict_proba(X_verified[val_idx])

    classes = model.named_steps["svm"].classes_

    predicted_indices = np.argmax(probabilities, axis=1)

    oof_predictions[val_idx] = classes[predicted_indices]

    oof_confidence[val_idx] = np.max(probabilities, axis=1)


# Compare predictions against the verified ground-truth labels
oof_correct = (oof_predictions == y_verified)

print("Verified samples:", len(y_verified))
print("OOF top-1 accuracy:", round(oof_correct.mean(), 4))


# Group samples by prediction confidence
verified_confidence_df = pd.DataFrame({
    "confidence": oof_confidence,
    "correct": oof_correct
})

verified_confidence_df["confidence_group"] = pd.cut(
    verified_confidence_df["confidence"],
    bins=[0, 0.4, 0.6, 0.8, 1.0],
    include_lowest=True
)

verified_confidence_summary = verified_confidence_df.groupby(
    "confidence_group",
    observed=False
).agg(
    sample_count=("correct", "count"),
    accuracy=("correct", "mean"),
    average_confidence=("confidence", "mean")
).reset_index()

display(verified_confidence_summary)


Verified samples: 620
OOF top-1 accuracy: 0.5774


,confidence_group,sample_count,accuracy,average_confidence
0,"(-0.001, 0.4]",495,0.476768,0.202790
1,"(0.4, 0.6]",77,0.961039,0.498515
2,"(0.6, 0.8]",46,1.000000,0.679744
3,"(0.8, 1.0]",2,1.000000,0.827750


Step 11 — Find a sensible confidence threshold

In [14]:

# Test different confidence thresholds on verified OOF predictions
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.60]

threshold_results = []

for threshold in thresholds:

    selected = oof_confidence >= threshold

    selected_count = np.sum(selected)

    if selected_count > 0:
        selected_accuracy = np.mean(oof_correct[selected])
    else:
        selected_accuracy = np.nan

    threshold_results.append({
        "Threshold": threshold,
        "Samples retained": selected_count,
        "Coverage (%)": round(
            selected_count / len(y_verified) * 100, 2
        ),
        "Top-1 accuracy (%)": round(
            selected_accuracy * 100, 2
        ) if selected_count > 0 else np.nan
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)


# Apply the same thresholds to unchecked samples
unchecked_threshold_results = []

for threshold in thresholds:

    selected = confidence_scores >= threshold

    unchecked_threshold_results.append({
        "Threshold": threshold,
        "Unchecked samples retained": int(np.sum(selected)),
        "Coverage (%)": round(
            np.mean(selected) * 100, 2
        ),
        "Agreement with existing labels (%)": round(
            unchecked_df.loc[
                selected, "agrees_with_existing_label"
            ].mean() * 100, 2
        ) if np.sum(selected) > 0 else np.nan
    })

unchecked_threshold_df = pd.DataFrame(
    unchecked_threshold_results
)

display(unchecked_threshold_df)


,Threshold,Samples retained,Coverage (%),Top-1 accuracy (%)
0,0.30,202,32.58,91.09
1,0.35,159,25.65,94.97
2,0.40,125,20.16,97.60
3,0.45,107,17.26,98.13
4,0.50,87,14.03,98.85
5,0.60,48,7.74,100.00


,Threshold,Unchecked samples retained,Coverage (%),Agreement with existing labels (%)
0,0.30,905,18.16,46.52
1,0.35,600,12.04,50.00
2,0.40,380,7.62,56.05
3,0.45,211,4.23,58.77
4,0.50,120,2.41,64.17
5,0.60,34,0.68,76.47


In [15]:

# Select unchecked samples with confidence >= 0.40
threshold = 0.40

selected_unchecked = unchecked_df[
    unchecked_df["confidence"] >= threshold
].copy()

# Count verified samples in each category
verified_counts = (
    pd.Series(y_verified)
    .value_counts()
    .rename("verified_count")
)

# Count high-confidence unchecked predictions in each category
pseudo_counts = (
    selected_unchecked["predicted_label"]
    .value_counts()
    .rename("high_confidence_count")
)

# Count all unchecked samples' original labels in each category
original_unchecked_counts = (
    unchecked_df["label"]
    .value_counts()
    .rename("original_unchecked_count")
)

# Combine the counts
class_coverage = pd.concat(
    [
        verified_counts,
        pseudo_counts,
        original_unchecked_counts
    ],
    axis=1
).fillna(0).astype(int)

class_coverage.index.name = "category"

class_coverage = class_coverage.sort_index()

display(class_coverage)

print("Total verified samples:", class_coverage["verified_count"].sum())
print("Total high-confidence unchecked samples:",
      class_coverage["high_confidence_count"].sum())


,verified_count,high_confidence_count,original_unchecked_count
category,,,
C01,10,21,162
C02,10,0,48
C03,40,36,2
C04,10,0,214
C05,40,27,40
C06,10,14,79
C07,10,26,206
C08,10,3,52
C09,10,12,171


Total verified samples: 620
Total high-confidence unchecked samples: 380


Step 12 — Test pseudo-labelling without validation leakage

In [16]:

# Step 12: Controlled pseudo-labelling experiment

threshold = 0.40
pseudo_weight = 0.25

baseline_oof = np.zeros(
    (len(y_verified), len(np.unique(y_verified)))
)

augmented_oof = np.zeros_like(baseline_oof)

fold_results = []
pseudo_counts = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_verified, y_verified), start=1
):

    # Split verified data into training and validation portions
    X_fold_train = X_verified[train_idx]
    y_fold_train = y_verified[train_idx]

    X_fold_val = X_verified[val_idx]
    y_fold_val = y_verified[val_idx]

    # -------------------------------------------------
    # 1. Train initial model using verified data only
    # -------------------------------------------------

    initial_model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            C=10.0,
            kernel="rbf",
            gamma="scale",
            class_weight=None,
            probability=True,
            random_state=42
        ))
    ])

    initial_model.fit(X_fold_train, y_fold_train)

    # Baseline predictions on held-out verified samples
    baseline_prob = initial_model.predict_proba(X_fold_val)

    baseline_classes = initial_model.named_steps["svm"].classes_

    baseline_oof[val_idx] = baseline_prob

    baseline_score = calculate_map3(
        y_fold_val,
        baseline_prob,
        baseline_classes
    )

    # -------------------------------------------------
    # 2. Generate pseudo-labels for unchecked samples
    # -------------------------------------------------

    unchecked_prob = initial_model.predict_proba(X_unchecked)

    predicted_indices = np.argmax(unchecked_prob, axis=1)

    pseudo_labels = baseline_classes[predicted_indices]

    confidence = np.max(unchecked_prob, axis=1)

    selected = confidence >= threshold

    X_pseudo = X_unchecked[selected]
    y_pseudo = pseudo_labels[selected]

    pseudo_counts.append(int(np.sum(selected)))

    # -------------------------------------------------
    # 3. Combine verified and pseudo-labelled samples
    # -------------------------------------------------

    X_augmented = np.vstack([
        X_fold_train,
        X_pseudo
    ])

    y_augmented = np.concatenate([
        y_fold_train,
        y_pseudo
    ])

    # Verified samples receive weight 1.0.
    # Pseudo-labelled samples receive lower weight 0.25.
    sample_weights = np.concatenate([
        np.ones(len(y_fold_train)),
        np.full(len(y_pseudo), pseudo_weight)
    ])

    # -------------------------------------------------
    # 4. Train augmented SVM
    # -------------------------------------------------

    augmented_model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            C=10.0,
            kernel="rbf",
            gamma="scale",
            class_weight=None,
            probability=True,
            random_state=42
        ))
    ])

    augmented_model.fit(
        X_augmented,
        y_augmented,
        svm__sample_weight=sample_weights
    )

    # Evaluate on exactly the same held-out verified fold
    augmented_prob = augmented_model.predict_proba(X_fold_val)

    augmented_classes = augmented_model.named_steps["svm"].classes_

    augmented_oof[val_idx] = augmented_prob

    augmented_score = calculate_map3(
        y_fold_val,
        augmented_prob,
        augmented_classes
    )

    fold_results.append({
        "Fold": fold,
        "Baseline MAP@3": baseline_score,
        "Augmented MAP@3": augmented_score,
        "Pseudo-labels added": int(np.sum(selected))
    })

# -------------------------------------------------
# 5. Compare overall OOF performance
# -------------------------------------------------

baseline_overall = calculate_map3(
    y_verified,
    baseline_oof,
    np.unique(y_verified)
)

augmented_overall = calculate_map3(
    y_verified,
    augmented_oof,
    np.unique(y_verified)
)

print("Baseline overall OOF MAP@3:", round(baseline_overall, 6))
print("Augmented overall OOF MAP@3:", round(augmented_overall, 6))
print("Difference:", round(
    augmented_overall - baseline_overall, 6
))

print("\nPseudo-labels selected in each fold:", pseudo_counts)
print("Average pseudo-labels per fold:", round(np.mean(pseudo_counts), 1))

display(pd.DataFrame(fold_results))


Baseline overall OOF MAP@3: 0.670699
Augmented overall OOF MAP@3: 0.671505
Difference: 0.000806

Pseudo-labels selected in each fold: [215, 146, 134, 153, 220]
Average pseudo-labels per fold: 173.6


,Fold,Baseline MAP@3,Augmented MAP@3,Pseudo-labels added
0,1,0.651882,0.645161,215
1,2,0.666667,0.678763,146
2,3,0.713710,0.701613,134
3,4,0.666667,0.665323,153
4,5,0.654570,0.666667,220


Pseudo-labelling weight experiment

In [17]:

# STEP 13: Complete pseudo-labelling weight comparison

pseudo_weights = [0.10, 0.25, 0.50, 1.00]
confidence_threshold = 0.40

weight_results = []

# Store results for every pseudo-label weight
for pseudo_weight in pseudo_weights:

    print(f"\nTesting pseudo-label weight: {pseudo_weight}")

    augmented_oof = np.zeros(
        (len(y_verified), len(np.unique(y_verified)))
    )

    fold_scores = []
    pseudo_counts = []

    # Five-fold cross-validation
    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X_verified, y_verified), start=1
    ):

        # Split verified samples
        X_fold_train = X_verified[train_idx]
        y_fold_train = y_verified[train_idx]

        X_fold_val = X_verified[val_idx]
        y_fold_val = y_verified[val_idx]

        # ---------------------------------------
        # 1. Train initial SVM on verified samples
        # ---------------------------------------

        initial_model = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(
                C=10.0,
                kernel="rbf",
                gamma="scale",
                class_weight=None,
                probability=True,
                random_state=42
            ))
        ])

        initial_model.fit(X_fold_train, y_fold_train)

        # ---------------------------------------
        # 2. Generate pseudo-labels
        # ---------------------------------------

        unchecked_prob = initial_model.predict_proba(X_unchecked)

        classes = initial_model.named_steps["svm"].classes_

        predicted_indices = np.argmax(unchecked_prob, axis=1)

        pseudo_labels = classes[predicted_indices]

        confidence = np.max(unchecked_prob, axis=1)

        selected = confidence >= confidence_threshold

        X_pseudo = X_unchecked[selected]
        y_pseudo = pseudo_labels[selected]

        pseudo_counts.append(len(y_pseudo))

        # ---------------------------------------
        # 3. Combine verified and pseudo samples
        # ---------------------------------------

        X_augmented = np.vstack([
            X_fold_train,
            X_pseudo
        ])

        y_augmented = np.concatenate([
            y_fold_train,
            y_pseudo
        ])

        sample_weights = np.concatenate([
            np.ones(len(y_fold_train)),
            np.full(len(y_pseudo), pseudo_weight)
        ])

        # ---------------------------------------
        # 4. Train augmented SVM
        # ---------------------------------------

        augmented_model = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(
                C=10.0,
                kernel="rbf",
                gamma="scale",
                class_weight=None,
                probability=True,
                random_state=42
            ))
        ])

        augmented_model.fit(
            X_augmented,
            y_augmented,
            svm__sample_weight=sample_weights
        )

        # ---------------------------------------
        # 5. Evaluate on held-out verified samples
        # ---------------------------------------

        val_prob = augmented_model.predict_proba(X_fold_val)

        augmented_oof[val_idx] = val_prob

        score = calculate_map3(
            y_fold_val,
            val_prob,
            augmented_model.named_steps["svm"].classes_
        )

        fold_scores.append(score)

    # ---------------------------------------
    # 6. Calculate overall OOF MAP@3
    # ---------------------------------------

    overall_score = calculate_map3(
        y_verified,
        augmented_oof,
        np.unique(y_verified)
    )

    weight_results.append({
        "Pseudo-label weight": pseudo_weight,
        "Mean Fold MAP@3": np.mean(fold_scores),
        "Overall OOF MAP@3": overall_score,
        "Fold Scores": np.round(fold_scores, 4).tolist(),
        "Pseudo-label counts": pseudo_counts,
        "Average pseudo-labels": round(np.mean(pseudo_counts), 1)
    })

# -------------------------------------------
# 7. Display comparison
# -------------------------------------------

weight_results_df = pd.DataFrame(weight_results)

weight_results_df = weight_results_df.sort_values(
    "Overall OOF MAP@3",
    ascending=False
).reset_index(drop=True)

print("\nPSEUDO-LABELLING WEIGHT COMPARISON")

display(weight_results_df)

# Compare against verified-only baseline
baseline_score = 0.670699

print("\nVERIFIED-ONLY BASELINE:", baseline_score)

weight_results_df["Improvement over baseline"] = (
    weight_results_df["Overall OOF MAP@3"] - baseline_score
)

display(weight_results_df[
    ["Pseudo-label weight", "Overall OOF MAP@3",
     "Improvement over baseline"]
])



Testing pseudo-label weight: 0.1

Testing pseudo-label weight: 0.25

Testing pseudo-label weight: 0.5

Testing pseudo-label weight: 1.0

PSEUDO-LABELLING WEIGHT COMPARISON


,Pseudo-label weight,Mean Fold MAP@3,Overall OOF MAP@3,Fold Scores,Pseudo-label counts,Average pseudo-labels
0,0.25,0.671505,0.671505,"[0.6452, 0.6788, 0.7016, 0.6653, 0.6667]","[215, 146, 134, 153, 220]",173.6
1,0.10,0.671237,0.671237,"[0.6505, 0.6761, 0.7003, 0.6653, 0.664]","[215, 146, 134, 153, 220]",173.6
2,1.00,0.670699,0.670699,"[0.6452, 0.6774, 0.6976, 0.6653, 0.668]","[215, 146, 134, 153, 220]",173.6
3,0.50,0.670430,0.670430,"[0.6452, 0.6774, 0.6976, 0.6653, 0.6667]","[215, 146, 134, 153, 220]",173.6



VERIFIED-ONLY BASELINE: 0.670699


,Pseudo-label weight,Overall OOF MAP@3,Improvement over baseline
0,0.25,0.671505,8.063763e-04
1,0.10,0.671237,5.375591e-04
2,1.00,0.670699,-7.526882e-08
3,0.50,0.670430,-2.688925e-04


audio feature engineering

In [18]:

# STEP 14: Extract additional audio features

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import time
from pathlib import Path

train_audio_dir = Path(
    "/kaggle/input/competitions/project4/audio/audio/train"
)

test_audio_dir = Path(
    "/kaggle/input/competitions/project4/audio/audio/test"
)

def extract_additional_features(file_path):

    y, sr = sf.read(file_path)

    # Convert to floating-point format
    y = y.astype(np.float32)

    # MFCCs
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=20,
        n_fft=2048,
        hop_length=512
    )

    # First-order and second-order MFCC deltas
    delta_mfcc = librosa.feature.delta(mfcc, order=1)
    delta2_mfcc = librosa.feature.delta(mfcc, order=2)

    # Spectral contrast
    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=sr,
        n_fft=2048,
        hop_length=512
    )

    features = np.concatenate([
        np.mean(delta_mfcc, axis=1),
        np.std(delta_mfcc, axis=1),
        np.mean(delta2_mfcc, axis=1),
        np.std(delta2_mfcc, axis=1),
        np.mean(contrast, axis=1),
        np.std(contrast, axis=1)
    ])

    return features


def extract_dataset_features(ids, audio_dir, dataset_name):

    all_features = []

    start = time.time()

    for i, audio_id in enumerate(ids):

        file_path = audio_dir / f"{audio_id}.wav"

        features = extract_additional_features(file_path)

        all_features.append(features)

        if (i + 1) % 500 == 0:
            print(
                dataset_name,
                i + 1,
                "/",
                len(ids),
                "completed"
            )

    all_features = np.array(all_features)

    print(
        dataset_name,
        "shape:",
        all_features.shape
    )

    print(
        dataset_name,
        "time:",
        round(time.time() - start, 2),
        "seconds"
    )

    return all_features


# Extract additional features for training recordings
X_train_additional = extract_dataset_features(
    train_df["id"].tolist(),
    train_audio_dir,
    "Training"
)

# Extract additional features for test recordings
X_test_additional = extract_dataset_features(
    test_df["id"].tolist(),
    test_audio_dir,
    "Testing"
)

# Save the extracted features
np.savez_compressed(
    "/kaggle/working/additional_audio_features.npz",
    X_train_additional=X_train_additional,
    X_test_additional=X_test_additional
)

print("\nAdditional feature extraction completed.")
print("Training features:", X_train_additional.shape)
print("Testing features:", X_test_additional.shape)


Training 500 / 5604 completed
Training 1000 / 5604 completed
Training 1500 / 5604 completed
Training 2000 / 5604 completed
Training 2500 / 5604 completed
Training 3000 / 5604 completed
Training 3500 / 5604 completed
Training 4000 / 5604 completed
Training 4500 / 5604 completed
Training 5000 / 5604 completed
Training 5500 / 5604 completed
Training shape: (5604, 94)
Training time: 214.59 seconds
Testing 500 / 1592 completed
Testing 1000 / 1592 completed
Testing 1500 / 1592 completed
Testing shape: (1592, 94)
Testing time: 47.64 seconds

Additional feature extraction completed.
Training features: (5604, 94)
Testing features: (1592, 94)


Step 15 — Align the new features and create feature combinations

In [19]:

# STEP 15: Align and prepare feature combinations

# Verify feature dimensions
print("Original training:", X_train_aligned.shape)
print("Additional training:", X_train_additional.shape)

print("Original testing:", X_test_aligned.shape)
print("Additional testing:", X_test_additional.shape)

# Check for invalid values
print("\nAdditional training NaNs:",
      np.isnan(X_train_additional).sum())

print("Additional training infinities:",
      np.isinf(X_train_additional).sum())

print("Additional testing NaNs:",
      np.isnan(X_test_additional).sum())

print("Additional testing infinities:",
      np.isinf(X_test_additional).sum())


# Create the three feature configurations
X_original_verified = X_train_aligned[verified_mask]

X_additional_verified = X_train_additional[verified_mask]

X_combined = np.hstack([
    X_train_aligned,
    X_train_additional
])

X_combined_verified = X_combined[verified_mask]


# Create corresponding test combinations
X_combined_test = np.hstack([
    X_test_aligned,
    X_test_additional
])


# Store configurations for later experiments
feature_sets = {
    "Original_75": X_original_verified,
    "Additional_94": X_additional_verified,
    "Combined_169": X_combined_verified
}

print("\nFeature configurations:")

for name, features in feature_sets.items():
    print(name, features.shape)


Original training: (5604, 75)
Additional training: (5604, 94)
Original testing: (1592, 75)
Additional testing: (1592, 94)

Additional training NaNs: 0
Additional training infinities: 0
Additional testing NaNs: 0
Additional testing infinities: 0

Feature configurations:
Original_75 (620, 75)
Additional_94 (620, 94)
Combined_169 (620, 169)


Step 16 — Compare all three feature configurations

In [20]:

# STEP 16: Compare audio feature configurations

feature_results = []

feature_oof_results = {}

for feature_name, X_features in feature_sets.items():

    print("\nEvaluating:", feature_name)

    oof_probabilities = np.zeros(
        (len(y_verified), len(np.unique(y_verified)))
    )

    fold_scores = []

    start_time = time.time()

    for train_idx, val_idx in cv.split(
        X_features, y_verified
    ):

        X_fold_train = X_features[train_idx]
        y_fold_train = y_verified[train_idx]

        X_fold_val = X_features[val_idx]
        y_fold_val = y_verified[val_idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(
                C=10.0,
                kernel="rbf",
                gamma="scale",
                class_weight=None,
                probability=True,
                random_state=42
            ))
        ])

        model.fit(X_fold_train, y_fold_train)

        probabilities = model.predict_proba(X_fold_val)

        oof_probabilities[val_idx] = probabilities

        score = calculate_map3(
            y_fold_val,
            probabilities,
            model.named_steps["svm"].classes_
        )

        fold_scores.append(score)

    overall_score = calculate_map3(
        y_verified,
        oof_probabilities,
        np.unique(y_verified)
    )

    elapsed_time = time.time() - start_time

    feature_oof_results[feature_name] = oof_probabilities

    feature_results.append({
        "Feature Set": feature_name,
        "Number of Features": X_features.shape[1],
        "Mean Fold MAP@3": np.mean(fold_scores),
        "Overall OOF MAP@3": overall_score,
        "Fold Scores": np.round(fold_scores, 4).tolist(),
        "Time (seconds)": round(elapsed_time, 2)
    })

feature_results_df = pd.DataFrame(feature_results)

feature_results_df = feature_results_df.sort_values(
    "Overall OOF MAP@3",
    ascending=False
).reset_index(drop=True)

print("\nFEATURE CONFIGURATION COMPARISON")

display(feature_results_df)



Evaluating: Original_75

Evaluating: Additional_94

Evaluating: Combined_169

FEATURE CONFIGURATION COMPARISON


,Feature Set,Number of Features,Mean Fold MAP@3,Overall OOF MAP@3,Fold Scores,Time (seconds)
0,Original_75,75,0.670699,0.670699,"[0.6519, 0.6667, 0.7137, 0.6667, 0.6546]",1.51
1,Combined_169,169,0.650000,0.650000,"[0.629, 0.621, 0.6815, 0.6317, 0.6868]",2.17
2,Additional_94,94,0.555376,0.555376,"[0.5497, 0.5323, 0.5565, 0.5551, 0.5833]",1.66


Step 17 — Identify which additional features are hurting or helping

In [21]:

# STEP 17: Individual additional feature group comparison

# Split the 94 additional features into their groups
X_delta = X_train_additional[:, :40]

X_delta2 = X_train_additional[:, 40:80]

X_contrast = X_train_additional[:, 80:94]

# Create feature configurations using all 5604 training samples
feature_experiments = {
    "Original_75": X_train_aligned,

    "Original_plus_Delta": np.hstack([
        X_train_aligned,
        X_delta
    ]),

    "Original_plus_Delta2": np.hstack([
        X_train_aligned,
        X_delta2
    ]),

    "Original_plus_Contrast": np.hstack([
        X_train_aligned,
        X_contrast
    ]),

    "Original_plus_All": np.hstack([
        X_train_aligned,
        X_train_additional
    ])
}

group_results = []

for feature_name, X_all in feature_experiments.items():

    # Use only verified samples for cross-validation
    X_features = X_all[verified_mask]

    oof_probabilities = np.zeros(
        (len(y_verified), len(np.unique(y_verified)))
    )

    fold_scores = []

    start_time = time.time()

    for train_idx, val_idx in cv.split(
        X_features, y_verified
    ):

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(
                C=10.0,
                kernel="rbf",
                gamma="scale",
                class_weight=None,
                probability=True,
                random_state=42
            ))
        ])

        model.fit(
            X_features[train_idx],
            y_verified[train_idx]
        )

        probabilities = model.predict_proba(
            X_features[val_idx]
        )

        oof_probabilities[val_idx] = probabilities

        score = calculate_map3(
            y_verified[val_idx],
            probabilities,
            model.named_steps["svm"].classes_
        )

        fold_scores.append(score)

    overall_score = calculate_map3(
        y_verified,
        oof_probabilities,
        np.unique(y_verified)
    )

    group_results.append({
        "Feature Set": feature_name,
        "Number of Features": X_features.shape[1],
        "Mean Fold MAP@3": np.mean(fold_scores),
        "Overall OOF MAP@3": overall_score,
        "Fold Scores": np.round(fold_scores, 4).tolist(),
        "Time (seconds)": round(time.time() - start_time, 2)
    })

group_results_df = pd.DataFrame(group_results)

group_results_df = group_results_df.sort_values(
    "Overall OOF MAP@3",
    ascending=False
).reset_index(drop=True)

display(group_results_df)


,Feature Set,Number of Features,Mean Fold MAP@3,Overall OOF MAP@3,Fold Scores,Time (seconds)
0,Original_plus_Contrast,89,0.690860,0.690860,"[0.6761, 0.6989, 0.7204, 0.6801, 0.6788]",1.55
1,Original_75,75,0.670699,0.670699,"[0.6519, 0.6667, 0.7137, 0.6667, 0.6546]",1.50
2,Original_plus_All,169,0.650000,0.650000,"[0.629, 0.621, 0.6815, 0.6317, 0.6868]",2.15
3,Original_plus_Delta2,115,0.642204,0.642204,"[0.6062, 0.6169, 0.6774, 0.6492, 0.6613]",1.74
4,Original_plus_Delta,115,0.639785,0.639785,"[0.6425, 0.6263, 0.6492, 0.6317, 0.6492]",1.68


Tune SVM on the improved 89 features

In [22]:

# STEP 18: Tune SVM using original + spectral contrast features

# Construct the best feature set: 75 original + 14 contrast
X_best_all = np.hstack([
    X_train_aligned,
    X_train_additional[:, 80:94]
])

X_best_verified = X_best_all[verified_mask]

print("Best feature shape:", X_best_verified.shape)

C_values = [3.0, 10.0, 20.0, 30.0]
class_weights = [None, "balanced"]

svm_tuning_results = []

for C_value in C_values:

    for weight in class_weights:

        print(
            "\nTesting C =", C_value,
            "| class_weight =", weight
        )

        oof_probabilities = np.zeros(
            (len(y_verified), len(np.unique(y_verified)))
        )

        fold_scores = []

        start_time = time.time()

        for train_idx, val_idx in cv.split(
            X_best_verified, y_verified
        ):

            model = Pipeline([
                ("scaler", StandardScaler()),
                ("svm", SVC(
                    C=C_value,
                    kernel="rbf",
                    gamma="scale",
                    class_weight=weight,
                    probability=True,
                    random_state=42
                ))
            ])

            model.fit(
                X_best_verified[train_idx],
                y_verified[train_idx]
            )

            probabilities = model.predict_proba(
                X_best_verified[val_idx]
            )

            oof_probabilities[val_idx] = probabilities

            score = calculate_map3(
                y_verified[val_idx],
                probabilities,
                model.named_steps["svm"].classes_
            )

            fold_scores.append(score)

        overall_score = calculate_map3(
            y_verified,
            oof_probabilities,
            np.unique(y_verified)
        )

        svm_tuning_results.append({
            "C": C_value,
            "class_weight": weight,
            "Mean Fold MAP@3": np.mean(fold_scores),
            "Overall OOF MAP@3": overall_score,
            "Fold Scores": np.round(fold_scores, 4).tolist(),
            "Time (seconds)": round(
                time.time() - start_time, 2
            )
        })

svm_tuning_df = pd.DataFrame(svm_tuning_results)

svm_tuning_df = svm_tuning_df.sort_values(
    "Overall OOF MAP@3",
    ascending=False
).reset_index(drop=True)

display(svm_tuning_df)

print("\nPrevious best verified-only score: 0.690860")


Best feature shape: (620, 89)

Testing C = 3.0 | class_weight = None

Testing C = 3.0 | class_weight = balanced

Testing C = 10.0 | class_weight = None

Testing C = 10.0 | class_weight = balanced

Testing C = 20.0 | class_weight = None

Testing C = 20.0 | class_weight = balanced

Testing C = 30.0 | class_weight = None

Testing C = 30.0 | class_weight = balanced


,C,class_weight,Mean Fold MAP@3,Overall OOF MAP@3,Fold Scores,Time (seconds)
0,3.0,balanced,0.694355,0.694355,"[0.6922, 0.6909, 0.7325, 0.6761, 0.6801]",1.54
1,10.0,balanced,0.691935,0.691935,"[0.6801, 0.6962, 0.7177, 0.6868, 0.6788]",1.55
2,3.0,None,0.691667,0.691667,"[0.6707, 0.6922, 0.7218, 0.6895, 0.6841]",1.56
3,30.0,balanced,0.691667,0.691667,"[0.6761, 0.6989, 0.7204, 0.6801, 0.6828]",1.53
4,10.0,None,0.690860,0.690860,"[0.6761, 0.6989, 0.7204, 0.6801, 0.6788]",1.54
5,20.0,None,0.690860,0.690860,"[0.6761, 0.6989, 0.7204, 0.6801, 0.6788]",1.52
6,30.0,None,0.690860,0.690860,"[0.6761, 0.6989, 0.7204, 0.6801, 0.6788]",1.53
7,20.0,balanced,0.689516,0.689516,"[0.6761, 0.6922, 0.7204, 0.6761, 0.6828]",1.54



Previous best verified-only score: 0.690860


Step 19 — Check XGBoost availability

In [23]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


Step 20 — XGBoost multiclass classification

In [24]:

# STEP 20: XGBoost model comparison

from xgboost import XGBClassifier

xgb_configs = [
    {
        "name": "XGB_depth3_lr03",
        "max_depth": 3,
        "learning_rate": 0.03,
        "n_estimators": 300
    },
    {
        "name": "XGB_depth4_lr03",
        "max_depth": 4,
        "learning_rate": 0.03,
        "n_estimators": 300
    },
    {
        "name": "XGB_depth3_lr05",
        "max_depth": 3,
        "learning_rate": 0.05,
        "n_estimators": 300
    }
]

# Convert category labels into integer class indices
class_labels = np.unique(y_verified)

label_to_index = {
    label: index
    for index, label in enumerate(class_labels)
}

y_verified_encoded = np.array([
    label_to_index[label]
    for label in y_verified
])

xgb_results = []

for config in xgb_configs:

    print("\nEvaluating:", config["name"])

    oof_probabilities = np.zeros(
        (len(y_verified), len(class_labels))
    )

    fold_scores = []

    start_time = time.time()

    for train_idx, val_idx in cv.split(
        X_best_verified, y_verified
    ):

        X_fold_train = X_best_verified[train_idx]
        X_fold_val = X_best_verified[val_idx]

        y_fold_train = y_verified_encoded[train_idx]
        y_fold_val = y_verified[val_idx]

        model = XGBClassifier(
            objective="multi:softprob",
            num_class=len(class_labels),
            max_depth=config["max_depth"],
            learning_rate=config["learning_rate"],
            n_estimators=config["n_estimators"],
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=-1,
            tree_method="hist"
        )

        model.fit(
            X_fold_train,
            y_fold_train
        )

        probabilities = model.predict_proba(X_fold_val)

        oof_probabilities[val_idx] = probabilities

        score = calculate_map3(
            y_fold_val,
            probabilities,
            class_labels
        )

        fold_scores.append(score)

    overall_score = calculate_map3(
        y_verified,
        oof_probabilities,
        class_labels
    )

    xgb_results.append({
        "Model": config["name"],
        "Mean Fold MAP@3": np.mean(fold_scores),
        "Overall OOF MAP@3": overall_score,
        "Fold Scores": np.round(fold_scores, 4).tolist(),
        "Time (seconds)": round(
            time.time() - start_time, 2
        )
    })

xgb_results_df = pd.DataFrame(xgb_results)

xgb_results_df = xgb_results_df.sort_values(
    "Overall OOF MAP@3",
    ascending=False
).reset_index(drop=True)

display(xgb_results_df)

print("\nCurrent best SVM MAP@3: 0.694355")



Evaluating: XGB_depth3_lr03

Evaluating: XGB_depth4_lr03

Evaluating: XGB_depth3_lr05


,Model,Mean Fold MAP@3,Overall OOF MAP@3,Fold Scores,Time (seconds)
0,XGB_depth3_lr05,0.649731,0.649731,"[0.6022, 0.6425, 0.6626, 0.6653, 0.6761]",55.69
1,XGB_depth4_lr03,0.640860,0.640860,"[0.5954, 0.6384, 0.6425, 0.6559, 0.672]",77.55
2,XGB_depth3_lr03,0.634140,0.634140,"[0.5914, 0.6344, 0.6384, 0.6478, 0.6586]",70.26



Current best SVM MAP@3: 0.694355


CatBoost

In [25]:
import catboost
from catboost import CatBoostClassifier

print("CatBoost version:", catboost.__version__)

CatBoost version: 1.2.10


Step 21 — CatBoost model comparison

In [26]:

# STEP 21: Faster CatBoost comparison

cat_configs = [
    {"name": "CatBoost_depth4", "iterations": 200, "depth": 4},
    {"name": "CatBoost_depth6", "iterations": 200, "depth": 6}
]

cat_results = []

for config in cat_configs:

    print("\nEvaluating:", config["name"])

    oof_probabilities = np.zeros(
        (len(y_verified), len(np.unique(y_verified)))
    )

    fold_scores = []
    start_time = time.time()

    for train_idx, val_idx in cv.split(
        X_best_verified, y_verified
    ):

        model = CatBoostClassifier(
            iterations=config["iterations"],
            depth=config["depth"],
            learning_rate=0.05,
            loss_function="MultiClass",
            verbose=False,
            random_seed=42,
            thread_count=-1,
            allow_writing_files=False
        )

        model.fit(
            X_best_verified[train_idx],
            y_verified_encoded[train_idx]
        )

        probabilities = model.predict_proba(
            X_best_verified[val_idx]
        )

        oof_probabilities[val_idx] = probabilities

        score = calculate_map3(
            y_verified[val_idx],
            probabilities,
            np.unique(y_verified)
        )

        fold_scores.append(score)

    overall_score = calculate_map3(
        y_verified,
        oof_probabilities,
        np.unique(y_verified)
    )

    cat_results.append({
        "Model": config["name"],
        "Overall OOF MAP@3": overall_score,
        "Fold Scores": np.round(fold_scores, 4).tolist(),
        "Time (seconds)": round(time.time() - start_time, 2)
    })

cat_results_df = pd.DataFrame(cat_results)

display(
    cat_results_df.sort_values(
        "Overall OOF MAP@3",
        ascending=False
    ).reset_index(drop=True)
)

print("Current best SVM MAP@3: 0.694355")



Evaluating: CatBoost_depth4

Evaluating: CatBoost_depth6


,Model,Overall OOF MAP@3,Fold Scores,Time (seconds)
0,CatBoost_depth6,0.651075,"[0.6559, 0.6478, 0.6774, 0.6653, 0.6089]",321.61
1,CatBoost_depth4,0.641667,"[0.6263, 0.6277, 0.6707, 0.6371, 0.6465]",73.56


Current best SVM MAP@3: 0.694355


Pseudo-labelling with 89 features

In [27]:

import numpy as np
import pandas as pd
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Current best SVM configuration
C_VALUE = 3
CONFIDENCE_THRESHOLD = 0.40

# Pseudo-label weights to compare
pseudo_weights = [0.10, 0.25, 0.50]

# Classes in sorted order
classes = np.unique(y_verified)

# Separate unchecked samples
X_unchecked = X_best_all[~verified_mask]

# Five-fold validation
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Store out-of-fold predictions
oof_predictions = {
    "Verified_only": np.zeros((len(y_verified), len(classes)))
}

for weight in pseudo_weights:
    oof_predictions[f"Pseudo_weight_{weight}"] = np.zeros(
        (len(y_verified), len(classes))
    )

fold_scores = {
    name: [] for name in oof_predictions
}

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_best_verified, y_verified), 1
):

    print(f"\nProcessing fold {fold}/5")

    X_fold_train = X_best_verified[train_idx]
    y_fold_train = y_verified[train_idx]

    X_fold_val = X_best_verified[val_idx]
    y_fold_val = y_verified[val_idx]

    # Initial model trained ONLY on verified fold-training samples
    initial_model = make_pipeline(
        StandardScaler(),
        SVC(
            C=C_VALUE,
            kernel="rbf",
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=42
        )
    )

    initial_model.fit(X_fold_train, y_fold_train)

    # Baseline predictions
    baseline_probs = initial_model.predict_proba(X_fold_val)

    oof_predictions["Verified_only"][val_idx] = baseline_probs

    baseline_score = calculate_map3(
        y_fold_val, baseline_probs, classes
    )

    fold_scores["Verified_only"].append(baseline_score)

    # Generate pseudo-labels for unchecked samples
    unchecked_probs = initial_model.predict_proba(X_unchecked)

    pseudo_confidence = np.max(unchecked_probs, axis=1)
    pseudo_indices = np.where(
        pseudo_confidence >= CONFIDENCE_THRESHOLD
    )[0]

    X_pseudo = X_unchecked[pseudo_indices]

    pseudo_labels = classes[
        np.argmax(unchecked_probs[pseudo_indices], axis=1)
    ]

    print(
        f"  Pseudo-labels selected: {len(pseudo_indices)}"
    )

    # Train one augmented model for each pseudo-label weight
    for weight in pseudo_weights:

        X_augmented = np.vstack([
            X_fold_train,
            X_pseudo
        ])

        y_augmented = np.concatenate([
            y_fold_train,
            pseudo_labels
        ])

        sample_weights = np.concatenate([
            np.ones(len(y_fold_train)),
            np.full(len(pseudo_labels), weight)
        ])

        augmented_model = make_pipeline(
            StandardScaler(),
            SVC(
                C=C_VALUE,
                kernel="rbf",
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=42
            )
        )

        augmented_model.fit(
            X_augmented,
            y_augmented,
            svc__sample_weight=sample_weights
        )

        augmented_probs = augmented_model.predict_proba(X_fold_val)

        model_name = f"Pseudo_weight_{weight}"

        oof_predictions[model_name][val_idx] = augmented_probs

        score = calculate_map3(
            y_fold_val,
            augmented_probs,
            classes
        )

        fold_scores[model_name].append(score)

        print(f"  {model_name}: MAP@3 = {score:.4f}")

# Summarize results
results = []

for name, predictions in oof_predictions.items():

    overall_score = calculate_map3(
        y_verified,
        predictions,
        classes
    )

    results.append({
        "Model": name,
        "Overall OOF MAP@3": overall_score,
        "Fold Scores": np.round(fold_scores[name], 4).tolist()
    })

results_df = pd.DataFrame(results).sort_values(
    "Overall OOF MAP@3",
    ascending=False
).reset_index(drop=True)

print("\nFINAL PSEUDO-LABELLING COMPARISON")
display(results_df)

print("\nCurrent best 89-feature SVM benchmark: 0.694355")
print(f"Total time: {time.time() - start_time:.2f} seconds")



Processing fold 1/5
  Pseudo-labels selected: 197
  Pseudo_weight_0.1: MAP@3 = 0.6989
  Pseudo_weight_0.25: MAP@3 = 0.6962
  Pseudo_weight_0.5: MAP@3 = 0.6895

Processing fold 2/5
  Pseudo-labels selected: 190
  Pseudo_weight_0.1: MAP@3 = 0.6734
  Pseudo_weight_0.25: MAP@3 = 0.6761
  Pseudo_weight_0.5: MAP@3 = 0.6815

Processing fold 3/5
  Pseudo-labels selected: 159
  Pseudo_weight_0.1: MAP@3 = 0.7204
  Pseudo_weight_0.25: MAP@3 = 0.7124
  Pseudo_weight_0.5: MAP@3 = 0.7151

Processing fold 4/5
  Pseudo-labels selected: 181
  Pseudo_weight_0.1: MAP@3 = 0.6788
  Pseudo_weight_0.25: MAP@3 = 0.6720
  Pseudo_weight_0.5: MAP@3 = 0.6694

Processing fold 5/5
  Pseudo-labels selected: 242
  Pseudo_weight_0.1: MAP@3 = 0.6841
  Pseudo_weight_0.25: MAP@3 = 0.6909
  Pseudo_weight_0.5: MAP@3 = 0.6976

FINAL PSEUDO-LABELLING COMPARISON


,Model,Overall OOF MAP@3,Fold Scores
0,Verified_only,0.694355,"[0.6922, 0.6909, 0.7325, 0.6761, 0.6801]"
1,Pseudo_weight_0.1,0.691129,"[0.6989, 0.6734, 0.7204, 0.6788, 0.6841]"
2,Pseudo_weight_0.5,0.690591,"[0.6895, 0.6815, 0.7151, 0.6694, 0.6976]"
3,Pseudo_weight_0.25,0.689516,"[0.6962, 0.6761, 0.7124, 0.672, 0.6909]"



Current best 89-feature SVM benchmark: 0.694355
Total time: 11.91 seconds


Experiment 2: Prediction Ensembling.

In [28]:

import numpy as np
import pandas as pd
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

# Current verified training data
X = X_best_verified
y = y_verified
classes = np.unique(y)
n_classes = len(classes)

# Five-fold validation
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Store out-of-fold probabilities for each model
model_names = ["SVM", "LogisticRegression", "RandomForest", "ExtraTrees"]

oof_probs = {
    name: np.zeros((len(y), n_classes))
    for name in model_names
}

# Align predicted probabilities to the common class order
def align_probabilities(model, X_data, classes):
    raw_probs = model.predict_proba(X_data)
    aligned = np.zeros((len(X_data), len(classes)))

    for i, class_label in enumerate(model.classes_):
        class_index = np.where(classes == class_label)[0][0]
        aligned[:, class_index] = raw_probs[:, i]

    return aligned


start_time = time.time()

# Train and evaluate all four models on identical folds
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

    print(f"\nProcessing fold {fold}/5")

    X_train_fold = X[train_idx]
    y_train_fold = y[train_idx]

    X_val_fold = X[val_idx]
    y_val_fold = y[val_idx]

    models = {
        "SVM": make_pipeline(
            StandardScaler(),
            SVC(
                C=3,
                kernel="rbf",
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=42
            )
        ),

        "LogisticRegression": make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=3000,
                random_state=42
            )
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=400,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ),

        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=400,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
    }

    for name, model in models.items():

        model.fit(X_train_fold, y_train_fold)

        probabilities = align_probabilities(
            model, X_val_fold, classes
        )

        oof_probs[name][val_idx] = probabilities

        score = calculate_map3(
            y_val_fold, probabilities, classes
        )

        print(f"  {name}: MAP@3 = {score:.4f}")


# Compare individual models and probability ensembles
ensemble_probs = {
    "SVM_only": oof_probs["SVM"],

    "SVM_75_LR_25":
        0.75 * oof_probs["SVM"] +
        0.25 * oof_probs["LogisticRegression"],

    "SVM_50_LR_50":
        0.50 * oof_probs["SVM"] +
        0.50 * oof_probs["LogisticRegression"],

    "SVM_75_RF_25":
        0.75 * oof_probs["SVM"] +
        0.25 * oof_probs["RandomForest"],

    "SVM_75_ET_25":
        0.75 * oof_probs["SVM"] +
        0.25 * oof_probs["ExtraTrees"],

    "SVM_75_Trees_25":
        0.75 * oof_probs["SVM"] +
        0.125 * oof_probs["RandomForest"] +
        0.125 * oof_probs["ExtraTrees"],

    "All_four_equal":
        0.25 * (
            oof_probs["SVM"] +
            oof_probs["LogisticRegression"] +
            oof_probs["RandomForest"] +
            oof_probs["ExtraTrees"]
        )
}

results = []

for name, probabilities in ensemble_probs.items():

    score = calculate_map3(y, probabilities, classes)

    results.append({
        "Model": name,
        "Overall OOF MAP@3": score
    })

results_df = pd.DataFrame(results).sort_values(
    "Overall OOF MAP@3",
    ascending=False
).reset_index(drop=True)

print("\nENSEMBLE COMPARISON")
display(results_df)

print("\nCurrent benchmark: 0.694355")
print(f"Total time: {time.time() - start_time:.2f} seconds")



Processing fold 1/5
  SVM: MAP@3 = 0.6922
  LogisticRegression: MAP@3 = 0.6532
  RandomForest: MAP@3 = 0.6720
  ExtraTrees: MAP@3 = 0.6505

Processing fold 2/5
  SVM: MAP@3 = 0.6909
  LogisticRegression: MAP@3 = 0.6989
  RandomForest: MAP@3 = 0.6882
  ExtraTrees: MAP@3 = 0.6909

Processing fold 3/5
  SVM: MAP@3 = 0.7325
  LogisticRegression: MAP@3 = 0.7151
  RandomForest: MAP@3 = 0.6949
  ExtraTrees: MAP@3 = 0.7204

Processing fold 4/5
  SVM: MAP@3 = 0.6761
  LogisticRegression: MAP@3 = 0.7016
  RandomForest: MAP@3 = 0.6841
  ExtraTrees: MAP@3 = 0.6976

Processing fold 5/5
  SVM: MAP@3 = 0.6801
  LogisticRegression: MAP@3 = 0.6532
  RandomForest: MAP@3 = 0.7083
  ExtraTrees: MAP@3 = 0.6962

ENSEMBLE COMPARISON


,Model,Overall OOF MAP@3
0,All_four_equal,0.713172
1,SVM_75_RF_25,0.710753
2,SVM_75_ET_25,0.710484
3,SVM_75_LR_25,0.708065
4,SVM_75_Trees_25,0.708065
5,SVM_50_LR_50,0.698656
6,SVM_only,0.694355



Current benchmark: 0.694355
Total time: 20.72 seconds



Step 22 — Robustness Validation

We evaluate whether the performance of our selected models is consistent across different cross-validation splits.

The purpose is to check whether our results depend heavily on a particular random seed.


In [29]:

import numpy as np
import pandas as pd
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

# Use the same 89-feature verified dataset
X = X_best_verified
y = y_verified
classes = np.unique(y)

# New validation splits
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

model_names = ["SVM", "LogisticRegression", "RandomForest", "ExtraTrees"]

oof_probs = {
    name: np.zeros((len(y), len(classes)))
    for name in model_names
}

def align_probabilities(model, X_data, classes):
    raw_probs = model.predict_proba(X_data)
    aligned = np.zeros((len(X_data), len(classes)))

    for i, class_label in enumerate(model.classes_):
        class_index = np.where(classes == class_label)[0][0]
        aligned[:, class_index] = raw_probs[:, i]

    return aligned

start_time = time.time()

# Train all four models on each new fold
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

    print(f"\nProcessing new fold {fold}/5")

    X_train_fold = X[train_idx]
    y_train_fold = y[train_idx]

    X_val_fold = X[val_idx]
    y_val_fold = y[val_idx]

    models = {
        "SVM": make_pipeline(
            StandardScaler(),
            SVC(
                C=3,
                kernel="rbf",
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=42
            )
        ),

        "LogisticRegression": make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=3000,
                random_state=42
            )
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=400,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ),

        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=400,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
    }

    for name, model in models.items():

        model.fit(X_train_fold, y_train_fold)

        probabilities = align_probabilities(
            model, X_val_fold, classes
        )

        oof_probs[name][val_idx] = probabilities

        score = calculate_map3(
            y_val_fold, probabilities, classes
        )

        print(f"  {name}: MAP@3 = {score:.4f}")


# Evaluate only our four shortlisted configurations
ensemble_probs = {
    "SVM_only": oof_probs["SVM"],

    "All_four_equal": (
        oof_probs["SVM"] +
        oof_probs["LogisticRegression"] +
        oof_probs["RandomForest"] +
        oof_probs["ExtraTrees"]
    ) / 4,

    "SVM_75_RF_25": (
        0.75 * oof_probs["SVM"] +
        0.25 * oof_probs["RandomForest"]
    ),

    "SVM_75_ET_25": (
        0.75 * oof_probs["SVM"] +
        0.25 * oof_probs["ExtraTrees"]
    )
}

results = []

for name, probabilities in ensemble_probs.items():

    overall_score = calculate_map3(
        y, probabilities, classes
    )

    fold_scores = []

    for _, val_idx in skf.split(X, y):
        fold_score = calculate_map3(
            y[val_idx],
            probabilities[val_idx],
            classes
        )
        fold_scores.append(round(fold_score, 4))

    results.append({
        "Model": name,
        "Overall OOF MAP@3": overall_score,
        "Fold Scores": fold_scores
    })

results_df = pd.DataFrame(results).sort_values(
    "Overall OOF MAP@3",
    ascending=False
).reset_index(drop=True)

print("\nROBUSTNESS VALIDATION RESULTS (SEED 2026)")
display(results_df)

print("\nPrevious seed 42 results:")
print("SVM only:          0.694355")
print("All four equal:    0.713172")
print("SVM + RF:          0.710753")
print("SVM + ExtraTrees:  0.710484")

print(f"\nTotal time: {time.time() - start_time:.2f} seconds")



Processing new fold 1/5
  SVM: MAP@3 = 0.6922
  LogisticRegression: MAP@3 = 0.6868
  RandomForest: MAP@3 = 0.6855
  ExtraTrees: MAP@3 = 0.6707

Processing new fold 2/5
  SVM: MAP@3 = 0.6895
  LogisticRegression: MAP@3 = 0.7245
  RandomForest: MAP@3 = 0.6828
  ExtraTrees: MAP@3 = 0.6801

Processing new fold 3/5
  SVM: MAP@3 = 0.6734
  LogisticRegression: MAP@3 = 0.7177
  RandomForest: MAP@3 = 0.6989
  ExtraTrees: MAP@3 = 0.6841

Processing new fold 4/5
  SVM: MAP@3 = 0.6801
  LogisticRegression: MAP@3 = 0.6694
  RandomForest: MAP@3 = 0.7204
  ExtraTrees: MAP@3 = 0.6895

Processing new fold 5/5
  SVM: MAP@3 = 0.6317
  LogisticRegression: MAP@3 = 0.6223
  RandomForest: MAP@3 = 0.6411
  ExtraTrees: MAP@3 = 0.6358

ROBUSTNESS VALIDATION RESULTS (SEED 2026)


,Model,Overall OOF MAP@3,Fold Scores
0,All_four_equal,0.704839,"[0.6949, 0.7366, 0.7298, 0.7016, 0.6613]"
1,SVM_75_ET_25,0.697312,"[0.6989, 0.7164, 0.6895, 0.7272, 0.6546]"
2,SVM_75_RF_25,0.693817,"[0.707, 0.7056, 0.6962, 0.707, 0.6532]"
3,SVM_only,0.673387,"[0.6922, 0.6895, 0.6734, 0.6801, 0.6317]"



Previous seed 42 results:
SVM only:          0.694355
All four equal:    0.713172
SVM + RF:          0.710753
SVM + ExtraTrees:  0.710484

Total time: 21.75 seconds


In [30]:

import numpy as np
import pandas as pd
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

# Current dataset: 89 features, verified samples only
X = X_best_verified
y = y_verified
classes = np.unique(y)

model_names = [
    "SVM",
    "LogisticRegression",
    "RandomForest",
    "ExtraTrees"
]

# Five controlled ensemble configurations.
# Order: SVM, Logistic Regression, Random Forest, Extra Trees
weight_configs = {
    "Equal_25_each": [0.25, 0.25, 0.25, 0.25],
    "SVM40_LR20_RF20_ET20": [0.40, 0.20, 0.20, 0.20],
    "SVM50_LR15_RF20_ET15": [0.50, 0.15, 0.20, 0.15],
    "SVM40_LR10_RF30_ET20": [0.40, 0.10, 0.30, 0.20],
    "SVM35_LR15_RF25_ET25": [0.35, 0.15, 0.25, 0.25]
}

seeds = [42, 2026]

# Align probability columns to the same class order
def align_probabilities(model, X_data, classes):
    raw_probs = model.predict_proba(X_data)
    aligned = np.zeros((len(X_data), len(classes)))

    for i, class_label in enumerate(model.classes_):
        class_index = np.where(classes == class_label)[0][0]
        aligned[:, class_index] = raw_probs[:, i]

    return aligned


all_seed_results = {}
start_time = time.time()

for seed in seeds:

    print(f"\n{'='*45}")
    print(f"VALIDATION SEED: {seed}")
    print(f"{'='*45}")

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    # Out-of-fold probabilities for each model
    oof_probs = {
        name: np.zeros((len(y), len(classes)))
        for name in model_names
    }

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X, y), 1
    ):

        print(f"\nSeed {seed} - Fold {fold}/5")

        X_train_fold = X[train_idx]
        y_train_fold = y[train_idx]

        X_val_fold = X[val_idx]
        y_val_fold = y[val_idx]

        models = {
            "SVM": make_pipeline(
                StandardScaler(),
                SVC(
                    C=3,
                    kernel="rbf",
                    gamma="scale",
                    class_weight="balanced",
                    probability=True,
                    random_state=42
                )
            ),

            "LogisticRegression": make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    C=1.0,
                    class_weight="balanced",
                    max_iter=3000,
                    random_state=42
                )
            ),

            "RandomForest": RandomForestClassifier(
                n_estimators=400,
                class_weight="balanced",
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            ),

            "ExtraTrees": ExtraTreesClassifier(
                n_estimators=400,
                class_weight="balanced",
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            )
        }

        for name, model in models.items():

            model.fit(X_train_fold, y_train_fold)

            oof_probs[name][val_idx] = align_probabilities(
                model, X_val_fold, classes
            )

            fold_score = calculate_map3(
                y_val_fold,
                oof_probs[name][val_idx],
                classes
            )

            print(f"  {name}: {fold_score:.4f}")

    # Evaluate the ensemble configurations on this seed
    seed_scores = {}

    for config_name, weights in weight_configs.items():

        combined_probs = sum(
            weight * oof_probs[model_name]
            for weight, model_name in zip(weights, model_names)
        )

        score = calculate_map3(y, combined_probs, classes)

        seed_scores[config_name] = score

    all_seed_results[seed] = seed_scores

    print(f"\nEnsemble results for seed {seed}:")
    display(
        pd.DataFrame([
            {"Configuration": name, "MAP@3": score}
            for name, score in seed_scores.items()
        ]).sort_values("MAP@3", ascending=False)
    )


# Combine results across both seeds
summary_rows = []

for config_name in weight_configs:

    score_42 = all_seed_results[42][config_name]
    score_2026 = all_seed_results[2026][config_name]

    summary_rows.append({
        "Configuration": config_name,
        "Seed 42": score_42,
        "Seed 2026": score_2026,
        "Mean MAP@3": (score_42 + score_2026) / 2
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    "Mean MAP@3",
    ascending=False
).reset_index(drop=True)

print("\nFINAL ENSEMBLE WEIGHT COMPARISON")
display(summary_df)

print("\nTotal time:", round(time.time() - start_time, 2), "seconds")



VALIDATION SEED: 42

Seed 42 - Fold 1/5
  SVM: 0.6922
  LogisticRegression: 0.6532
  RandomForest: 0.6720
  ExtraTrees: 0.6505

Seed 42 - Fold 2/5
  SVM: 0.6909
  LogisticRegression: 0.6989
  RandomForest: 0.6882
  ExtraTrees: 0.6909

Seed 42 - Fold 3/5
  SVM: 0.7325
  LogisticRegression: 0.7151
  RandomForest: 0.6949
  ExtraTrees: 0.7204

Seed 42 - Fold 4/5
  SVM: 0.6761
  LogisticRegression: 0.7016
  RandomForest: 0.6841
  ExtraTrees: 0.6976

Seed 42 - Fold 5/5
  SVM: 0.6801
  LogisticRegression: 0.6532
  RandomForest: 0.7083
  ExtraTrees: 0.6962

Ensemble results for seed 42:


,Configuration,MAP@3
2,SVM50_LR15_RF20_ET15,0.727419
3,SVM40_LR10_RF30_ET20,0.726344
4,SVM35_LR15_RF25_ET25,0.724731
1,SVM40_LR20_RF20_ET20,0.719892
0,Equal_25_each,0.713172



VALIDATION SEED: 2026

Seed 2026 - Fold 1/5
  SVM: 0.6922
  LogisticRegression: 0.6868
  RandomForest: 0.6855
  ExtraTrees: 0.6707

Seed 2026 - Fold 2/5
  SVM: 0.6895
  LogisticRegression: 0.7245
  RandomForest: 0.6828
  ExtraTrees: 0.6801

Seed 2026 - Fold 3/5
  SVM: 0.6734
  LogisticRegression: 0.7177
  RandomForest: 0.6989
  ExtraTrees: 0.6841

Seed 2026 - Fold 4/5
  SVM: 0.6801
  LogisticRegression: 0.6694
  RandomForest: 0.7204
  ExtraTrees: 0.6895

Seed 2026 - Fold 5/5
  SVM: 0.6317
  LogisticRegression: 0.6223
  RandomForest: 0.6411
  ExtraTrees: 0.6358

Ensemble results for seed 2026:


,Configuration,MAP@3
1,SVM40_LR20_RF20_ET20,0.710753
3,SVM40_LR10_RF30_ET20,0.710215
2,SVM50_LR15_RF20_ET15,0.708333
4,SVM35_LR15_RF25_ET25,0.707796
0,Equal_25_each,0.704839



FINAL ENSEMBLE WEIGHT COMPARISON


,Configuration,Seed 42,Seed 2026,Mean MAP@3
0,SVM40_LR10_RF30_ET20,0.726344,0.710215,0.718280
1,SVM50_LR15_RF20_ET15,0.727419,0.708333,0.717876
2,SVM35_LR15_RF25_ET25,0.724731,0.707796,0.716263
3,SVM40_LR20_RF20_ET20,0.719892,0.710753,0.715323
4,Equal_25_each,0.713172,0.704839,0.709005



Total time: 42.71 seconds


Third-seed confirmation experimenT

In [31]:

import numpy as np
import pandas as pd
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

# Same 89-feature verified dataset
X = X_best_verified
y = y_verified
classes = np.unique(y)

# Third validation seed
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=31415
)

model_names = [
    "SVM",
    "LogisticRegression",
    "RandomForest",
    "ExtraTrees"
]

oof_probs = {
    name: np.zeros((len(y), len(classes)))
    for name in model_names
}

def align_probabilities(model, X_data, classes):
    raw_probs = model.predict_proba(X_data)
    aligned = np.zeros((len(X_data), len(classes)))

    for i, class_label in enumerate(model.classes_):
        class_index = np.where(classes == class_label)[0][0]
        aligned[:, class_index] = raw_probs[:, i]

    return aligned

start_time = time.time()

# Train four models using identical folds
for fold, (train_idx, val_idx) in enumerate(
    skf.split(X, y), 1
):

    print(f"\nProcessing fold {fold}/5")

    X_train_fold = X[train_idx]
    y_train_fold = y[train_idx]

    X_val_fold = X[val_idx]
    y_val_fold = y[val_idx]

    models = {
        "SVM": make_pipeline(
            StandardScaler(),
            SVC(
                C=3,
                kernel="rbf",
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=42
            )
        ),

        "LogisticRegression": make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=3000,
                random_state=42
            )
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=400,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ),

        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=400,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
    }

    for name, model in models.items():

        model.fit(X_train_fold, y_train_fold)

        probabilities = align_probabilities(
            model, X_val_fold, classes
        )

        oof_probs[name][val_idx] = probabilities

        score = calculate_map3(
            y_val_fold, probabilities, classes
        )

        print(f"  {name}: MAP@3 = {score:.4f}")


# Compare the three frozen configurations
ensemble_probs = {
    "SVM_only": oof_probs["SVM"],

    "Equal_four": (
        oof_probs["SVM"] +
        oof_probs["LogisticRegression"] +
        oof_probs["RandomForest"] +
        oof_probs["ExtraTrees"]
    ) / 4,

    "Weighted_40_10_30_20": (
        0.40 * oof_probs["SVM"] +
        0.10 * oof_probs["LogisticRegression"] +
        0.30 * oof_probs["RandomForest"] +
        0.20 * oof_probs["ExtraTrees"]
    )
}

results = []

for name, probabilities in ensemble_probs.items():

    overall_score = calculate_map3(
        y, probabilities, classes
    )

    fold_scores = []

    for _, val_idx in skf.split(X, y):
        fold_score = calculate_map3(
            y[val_idx],
            probabilities[val_idx],
            classes
        )
        fold_scores.append(round(fold_score, 4))

    results.append({
        "Model": name,
        "Overall OOF MAP@3": overall_score,
        "Fold Scores": fold_scores
    })

results_df = pd.DataFrame(results).sort_values(
    "Overall OOF MAP@3",
    ascending=False
).reset_index(drop=True)

print("\nTHIRD-SEED CONFIRMATION RESULTS")
display(results_df)

print("\nPrevious weighted ensemble scores:")
print("Seed 42:   0.726344")
print("Seed 2026: 0.710215")

print(f"\nTotal time: {time.time() - start_time:.2f} seconds")



Processing fold 1/5
  SVM: MAP@3 = 0.7030
  LogisticRegression: MAP@3 = 0.6680
  RandomForest: MAP@3 = 0.6815
  ExtraTrees: MAP@3 = 0.6626

Processing fold 2/5
  SVM: MAP@3 = 0.6868
  LogisticRegression: MAP@3 = 0.6882
  RandomForest: MAP@3 = 0.6895
  ExtraTrees: MAP@3 = 0.6882

Processing fold 3/5
  SVM: MAP@3 = 0.7003
  LogisticRegression: MAP@3 = 0.6344
  RandomForest: MAP@3 = 0.7245
  ExtraTrees: MAP@3 = 0.7056

Processing fold 4/5
  SVM: MAP@3 = 0.7204
  LogisticRegression: MAP@3 = 0.7272
  RandomForest: MAP@3 = 0.7500
  ExtraTrees: MAP@3 = 0.7446

Processing fold 5/5
  SVM: MAP@3 = 0.6586
  LogisticRegression: MAP@3 = 0.6250
  RandomForest: MAP@3 = 0.6277
  ExtraTrees: MAP@3 = 0.6183

THIRD-SEED CONFIRMATION RESULTS


,Model,Overall OOF MAP@3,Fold Scores
0,Weighted_40_10_30_20,0.726075,"[0.7151, 0.7339, 0.7325, 0.7675, 0.6815]"
1,Equal_four,0.716129,"[0.7016, 0.7366, 0.7083, 0.7608, 0.6734]"
2,SVM_only,0.693817,"[0.703, 0.6868, 0.7003, 0.7204, 0.6586]"



Previous weighted ensemble scores:
Seed 42:   0.726344
Seed 2026: 0.710215

Total time: 21.97 seconds


Features: 89 (75 original + 14 spectral contrast)
Training data: 620 verified samples only
Models: SVM, Logistic Regression, Random Forest, Extra Trees
Ensemble weights: 40%, 10%, 30%, 20%
Development benchmark: 0.720878 mean OOF MAP@3 across three seeds

Train final models and generate test probabilities

In [32]:

import numpy as np
import pandas as pd
import time

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

start_time = time.time()

# --------------------------------------------------
# 1. Prepare final training and test feature arrays
# --------------------------------------------------

X_final_train = X_best_verified
y_final_train = y_verified

# Original 75 features + 14 spectral contrast features
X_final_test = np.hstack([
    X_test_aligned,
    X_test_additional[:, 80:94]
])

classes = np.unique(y_final_train)

print("Training data shape:", X_final_train.shape)
print("Test data shape:", X_final_test.shape)
print("Number of classes:", len(classes))

# Basic validation
assert X_final_train.shape[1] == 89
assert X_final_test.shape[1] == 89
assert len(X_final_train) == len(y_final_train)
assert len(classes) == 41
assert np.isfinite(X_final_train).all()
assert np.isfinite(X_final_test).all()

# --------------------------------------------------
# 2. Define the four selected models
# --------------------------------------------------

final_models = {
    "SVM": make_pipeline(
        StandardScaler(),
        SVC(
            C=3,
            kernel="rbf",
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=42
        )
    ),

    "LogisticRegression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        )
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=400,
        class_weight="balanced",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=400,
        class_weight="balanced",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

# --------------------------------------------------
# 3. Train each model and predict test probabilities
# --------------------------------------------------

final_test_probs = {}

for name, model in final_models.items():

    print(f"\nTraining {name}...")

    model.fit(X_final_train, y_final_train)

    raw_probs = model.predict_proba(X_final_test)

    # Align probability columns to the common class order
    aligned_probs = np.zeros((len(X_final_test), len(classes)))

    for i, class_label in enumerate(model.classes_):
        class_index = np.where(classes == class_label)[0][0]
        aligned_probs[:, class_index] = raw_probs[:, i]

    final_test_probs[name] = aligned_probs

    print("  Probability shape:", aligned_probs.shape)
    print("  Probability range:",
          aligned_probs.min(), "to", aligned_probs.max())

# --------------------------------------------------
# 4. Combine predictions using frozen weights
# --------------------------------------------------

final_ensemble_probs = (
    0.40 * final_test_probs["SVM"] +
    0.10 * final_test_probs["LogisticRegression"] +
    0.30 * final_test_probs["RandomForest"] +
    0.20 * final_test_probs["ExtraTrees"]
)

# --------------------------------------------------
# 5. Validate combined probabilities
# --------------------------------------------------

assert final_ensemble_probs.shape == (1592, 41)
assert np.isfinite(final_ensemble_probs).all()

row_sums = final_ensemble_probs.sum(axis=1)

assert np.allclose(row_sums, 1.0, atol=1e-6)

print("\nFINAL MODEL TRAINING COMPLETE")
print("Combined probability shape:", final_ensemble_probs.shape)
print("Number of test recordings:", len(final_ensemble_probs))
print("Number of categories:", len(classes))
print("All probabilities valid:", np.isfinite(final_ensemble_probs).all())
print("All probability rows sum to 1:", np.allclose(row_sums, 1.0))

print("\nTotal time:", round(time.time() - start_time, 2), "seconds")


Training data shape: (620, 89)
Test data shape: (1592, 89)
Number of classes: 41

Training SVM...
  Probability shape: (1592, 41)
  Probability range: 0.0006755179131320447 to 0.8517147306797503

Training LogisticRegression...
  Probability shape: (1592, 41)
  Probability range: 4.6791837500856444e-17 to 0.9994984613029625

Training RandomForest...
  Probability shape: (1592, 41)
  Probability range: 0.0 to 0.9450937534687534

Training ExtraTrees...
  Probability shape: (1592, 41)
  Probability range: 0.0 to 0.9370416666666667

FINAL MODEL TRAINING COMPLETE
Combined probability shape: (1592, 41)
Number of test recordings: 1592
Number of categories: 41
All probabilities valid: True
All probability rows sum to 1: True

Total time: 5.34 seconds


Inspect the submission format

In [33]:

# Load the official sample submission
sample_submission = pd.read_csv(
    "/kaggle/input/competitions/project4/sample_submission.csv"
)

print("Sample submission shape:", sample_submission.shape)

print("\nFirst 10 rows:")
display(sample_submission.head(10))

print("\nColumn names:")
print(sample_submission.columns.tolist())

print("\nExample label values:")
print(sample_submission["label"].head(10).tolist())

print("\nUnique label formats:")
print(sample_submission["label"].dropna().unique()[:10])


Sample submission shape: (1592, 2)

First 10 rows:


,id,label
0,c99ed9e0,C01 C02 C03
1,f7988da2,C01 C02 C03
2,af43e3af,C01 C02 C03
3,fbb23ad8,C01 C02 C03
4,3be2034b,C01 C02 C03
5,17a321bb,C01 C02 C03
6,58f8f967,C01 C02 C03
7,f6bc0df5,C01 C02 C03
8,b6612a08,C01 C02 C03
9,7cbdbc8d,C01 C02 C03



Column names:
['id', 'label']

Example label values:
['C01 C02 C03', 'C01 C02 C03', 'C01 C02 C03', 'C01 C02 C03', 'C01 C02 C03', 'C01 C02 C03', 'C01 C02 C03', 'C01 C02 C03', 'C01 C02 C03', 'C01 C02 C03']

Unique label formats:
['C01 C02 C03']


In [34]:

# --------------------------------------------------
# 1. Get the top 3 predicted categories
# --------------------------------------------------

top3_indices = np.argsort(
    final_ensemble_probs,
    axis=1
)[:, -3:][:, ::-1]

top3_labels = classes[top3_indices]

# Join the three category codes with spaces
predicted_labels = [
    " ".join(row)
    for row in top3_labels
]

# --------------------------------------------------
# 2. Create submission using official sample format
# --------------------------------------------------

submission = sample_submission.copy()

# Map predictions to test IDs
prediction_df = pd.DataFrame({
    "id": test_df["id"],
    "label": predicted_labels
})

# Preserve the exact ID order of sample_submission
submission = submission[["id"]].merge(
    prediction_df,
    on="id",
    how="left",
    validate="one_to_one",
    sort=False
)

# --------------------------------------------------
# 3. Validate the submission
# --------------------------------------------------

assert submission.shape == (1592, 2)
assert submission["id"].equals(sample_submission["id"])
assert submission["label"].notna().all()
assert submission["id"].is_unique

# Check that every row has exactly three distinct labels
for label_string in submission["label"]:

    labels = label_string.split()

    assert len(labels) == 3
    assert len(set(labels)) == 3
    assert all(label in classes for label in labels)

# --------------------------------------------------
# 4. Save the CSV
# --------------------------------------------------

submission_path = "/kaggle/working/submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("SUBMISSION FILE CREATED SUCCESSFULLY")
print("Shape:", submission.shape)
print("Columns:", submission.columns.tolist())

print("\nFirst 10 predictions:")
display(submission.head(10))

print("\nSaved at:", submission_path)
print("Total rows:", len(submission))
print("All IDs unique:", submission["id"].is_unique)
print("All labels contain exactly 3 categories:", True)


SUBMISSION FILE CREATED SUCCESSFULLY
Shape: (1592, 2)
Columns: ['id', 'label']

First 10 predictions:


,id,label
0,c99ed9e0,C39 C24 C32
1,f7988da2,C39 C30 C05
2,af43e3af,C39 C41 C40
3,fbb23ad8,C05 C04 C26
4,3be2034b,C03 C15 C27
5,17a321bb,C16 C38 C08
6,58f8f967,C20 C38 C31
7,f6bc0df5,C07 C24 C29
8,b6612a08,C40 C39 C26
9,7cbdbc8d,C19 C04 C05



Saved at: /kaggle/working/submission.csv
Total rows: 1592
All IDs unique: True
All labels contain exactly 3 categories: True


FINAL AUDIT

In [35]:

from pathlib import Path
import numpy as np
import pandas as pd

# 1. Reload the actual saved CSV
submission_check = pd.read_csv(
    "/kaggle/working/submission.csv"
)

# 2. Check file and structure
assert Path("/kaggle/working/submission.csv").exists()
assert submission_check.shape == (1592, 2)
assert submission_check.columns.tolist() == ["id", "label"]

# 3. Check IDs against both original files
assert submission_check["id"].equals(sample_submission["id"])
assert submission_check["id"].equals(test_df["id"])
assert submission_check["id"].is_unique

# 4. Recalculate expected top-three predictions
expected_indices = np.argsort(
    final_ensemble_probs,
    axis=1
)[:, -3:][:, ::-1]

expected_labels = classes[expected_indices]

expected_strings = [
    " ".join(row)
    for row in expected_labels
]

assert submission_check["label"].tolist() == expected_strings

# 5. Check label formatting
for label_string in submission_check["label"]:
    labels = label_string.split()

    assert len(labels) == 3
    assert len(set(labels)) == 3
    assert all(label in classes for label in labels)

# 6. Inspect category prediction frequency
predicted_category_counts = pd.Series(
    submission_check["label"].str.split().explode()
).value_counts().sort_index()

print("FINAL AUDIT PASSED")
print("File exists:", True)
print("Rows:", len(submission_check))
print("Columns:", submission_check.columns.tolist())
print("IDs match test.csv:", True)
print("Predictions match model probabilities:", True)
print("Exactly 3 valid labels per row:", True)

print("\nCategory prediction frequency:")
display(
    predicted_category_counts.rename("Prediction Count").to_frame()
)

print("\nFirst 10 final rows:")
display(submission_check.head(10))


FINAL AUDIT PASSED
File exists: True
Rows: 1592
Columns: ['id', 'label']
IDs match test.csv: True
Predictions match model probabilities: True
Exactly 3 valid labels per row: True

Category prediction frequency:


,Prediction Count
label,
C01,82
C02,62
C03,227
C04,153
C05,427
C06,63
C07,103
C08,86
C09,53



First 10 final rows:


,id,label
0,c99ed9e0,C39 C24 C32
1,f7988da2,C39 C30 C05
2,af43e3af,C39 C41 C40
3,fbb23ad8,C05 C04 C26
4,3be2034b,C03 C15 C27
5,17a321bb,C16 C38 C08
6,58f8f967,C20 C38 C31
7,f6bc0df5,C07 C24 C29
8,b6612a08,C40 C39 C26
9,7cbdbc8d,C19 C04 C05



Final Results and Conclusion

Final Approach
- **Feature set:** 89 audio features (75 original features + 14 spectral contrast features).
- **Training data:** 620 verified samples.
- **Models:** RBF SVM, Logistic Regression, Random Forest, and Extra Trees.
- **Ensemble weights:** 40%, 10%, 30%, and 20%, respectively.
- **Evaluation metric:** MAP@3.

Development Performance
The selected ensemble achieved a mean cross-validation MAP@3 of **0.720878 across three random seeds**.

Key Findings
- Spectral contrast features improved performance compared with the original 75-feature set.
- MFCC delta and delta-delta features did not improve the selected feature configuration.
- The verified labels were used for model training to avoid relying on potentially incorrect unchecked labels.
- Ensemble modelling provided a stronger development benchmark than the individual baseline experiments.

Final Prediction
The final ensemble generated predictions for all 1,592 test recordings. Each recording has three distinct category predictions, ordered by predicted probability.

The submission file was generated and passed the final structural and consistency checks.

